In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# READ-ONLY LATESTFAIL DIAGNOSTIC
# Project 16: apache@rocketmq
# Nothing in the frozen experiment is modified.
# ------------------------------------------------------------------

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
RESULTS_ROOT = THESIS_ROOT / "Results"

PROJECT_SLUG = "apache__rocketmq"
PROJECT_SHORT = "ROCKETMQ"

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

CONDITION_PLAN_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

RAW_RESULT_ROOT = (
    RESULTS_ROOT
    / "Raw"
    / PROJECT_SLUG
)

print("Condition plan:", CONDITION_PLAN_PATH)
print("Raw result root:", RAW_RESULT_ROOT)

assert CONDITION_PLAN_PATH.is_file(), CONDITION_PLAN_PATH
assert RAW_RESULT_ROOT.is_dir(), RAW_RESULT_ROOT

plan = pd.read_csv(CONDITION_PLAN_PATH)

# Use the SAME seed at 0% and 5%.
selected = plan[
    (plan["RepetitionSeed"] == 1)
    & (plan["NoisePercent"].isin([0, 5]))
].copy()

print("\nSelected conditions:")
display(
    selected[
        ["ConditionID", "NoisePercent", "RepetitionSeed"]
    ]
)

assert len(selected) == 2

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Condition plan: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/apache__rocketmq/ROCKETMQ_noise_plan/ROCKETMQ_condition_plan.csv
Raw result root: /content/drive/MyDrive/Thesis_Experiment/Results/Raw/apache__rocketmq

Selected conditions:


,ConditionID,NoisePercent,RepetitionSeed
0,noise_00__seed_01,0,1
1,noise_05__seed_01,5,1


In [2]:
def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)

    n = len(failures)
    m = int(failures.sum())

    if n == 0 or m == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - failure_positions.sum() / (n * m)
        + 1.0 / (2.0 * n)
    )


def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    if len(failures) == 0 or failures.sum() == 0:
        return np.nan

    total_duration = durations.sum()

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate(
        [np.array([0.0]), np.cumsum(durations)[:-1]]
    )

    failure_mask = failures == 1

    midpoint_times = (
        cumulative_before[failure_mask]
        + 0.5 * durations[failure_mask]
    )

    return float(
        1.0
        - np.mean(midpoint_times / total_duration)
    )


audit_rows = []
comparison_frames = []

for row in selected.itertuples(index=False):

    condition_id = str(row.ConditionID)
    noise = int(row.NoisePercent)

    ranking_path = (
        RAW_RESULT_ROOT
        / condition_id
        / "rankings.csv.gz"
    )

    assert ranking_path.is_file(), ranking_path

    rankings = pd.read_csv(ranking_path)

    lf = rankings[
        rankings["Technique"] == "LatestFail"
    ].copy()

    # Because original LatestFail score = -REC_LastFailureAge:
    # Score == +1 means REC_LastFailureAge == -1,
    # i.e. NO previous failure.
    lf["NeverFailed"] = np.isclose(
        lf["Score"].to_numpy(dtype=float),
        1.0,
        rtol=0,
        atol=1e-12,
    )

    sentinel_rows = int(lf["NeverFailed"].sum())
    total_rows = len(lf)

    sentinel_failures = int(
        lf.loc[
            lf["NeverFailed"],
            "CleanFailure"
        ].sum()
    )

    # --------------------------------------------------------------
    # CURRENT ranking
    # --------------------------------------------------------------
    current_metric_rows = []

    for build, g in lf.groupby("Build", sort=False):
        g = g.sort_values("Rank")

        if g["CleanFailure"].sum() == 0:
            continue

        current_metric_rows.append({
            "APFD": calculate_apfd(
                g["CleanFailure"]
            ),
            "APFDc": calculate_apfdc(
                g["CleanFailure"],
                g["Duration"],
            ),
        })

    current_metrics = pd.DataFrame(current_metric_rows)

    # --------------------------------------------------------------
    # CORRECTED interpretation:
    # tests that have NEVER failed must appear AFTER tests
    # that have an actual previous failure.
    #
    # Existing Score is still used among tests with real failures.
    # --------------------------------------------------------------
    corrected = (
        lf.sort_values(
            [
                "Build",
                "NeverFailed",
                "Score",
                "Test",
            ],
            ascending=[
                True,    # build
                True,    # False (has failed) before True (never failed)
                False,   # more recent actual failure first
                True,    # deterministic tie-break
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    corrected["CorrectedRank"] = (
        corrected
        .groupby("Build", sort=False)
        .cumcount()
        .add(1)
    )

    corrected_metric_rows = []

    for build, g in corrected.groupby("Build", sort=False):

        if g["CleanFailure"].sum() == 0:
            continue

        corrected_metric_rows.append({
            "APFD": calculate_apfd(
                g["CleanFailure"]
            ),
            "APFDc": calculate_apfdc(
                g["CleanFailure"],
                g["Duration"],
            ),
        })

    corrected_metrics = pd.DataFrame(
        corrected_metric_rows
    )

    audit_rows.append({
        "NoisePercent": noise,
        "Rows": total_rows,
        "NeverFailedRows": sentinel_rows,
        "NeverFailedPercent":
            100 * sentinel_rows / total_rows,
        "CleanFailuresInsideNeverFailedRows":
            sentinel_failures,

        "CurrentMeanAPFD":
            current_metrics["APFD"].mean(),
        "CorrectedMeanAPFD":
            corrected_metrics["APFD"].mean(),

        "CurrentMeanAPFDc":
            current_metrics["APFDc"].mean(),
        "CorrectedMeanAPFDc":
            corrected_metrics["APFDc"].mean(),
    })

    comparison_frames.append(
        corrected[
            [
                "NoisePercent",
                "RepetitionSeed",
                "Build",
                "Test",
                "Score",
                "NeverFailed",
                "Rank",
                "CorrectedRank",
                "CleanFailure",
                "Duration",
            ]
        ]
    )


audit = pd.DataFrame(audit_rows).sort_values(
    "NoisePercent"
)

print("\n=== LATESTFAIL SENTINEL AUDIT ===")
display(audit)

comparison = pd.concat(
    comparison_frames,
    ignore_index=True
)

print("\nExamples where ranking changes:")
changed = comparison[
    comparison["Rank"]
    != comparison["CorrectedRank"]
]

display(changed.head(50))


=== LATESTFAIL SENTINEL AUDIT ===


,NoisePercent,Rows,NeverFailedRows,NeverFailedPercent,CleanFailuresInsideNeverFailedRows,CurrentMeanAPFD,CorrectedMeanAPFD,CurrentMeanAPFDc,CorrectedMeanAPFDc
0,0,1641,1433,87.324802,0,0.103191,0.976398,0.299580,0.894954
1,5,1641,17,1.035954,0,0.798178,0.808288,0.754531,0.760465



Examples where ranking changes:


,NoisePercent,RepetitionSeed,Build,Test,Score,NeverFailed,Rank,CorrectedRank,CleanFailure,Duration
0,0,1,774660918,1170,-79.0,False,179,1,0,1073.0
1,0,1,774660918,1618,-79.0,False,180,2,0,1027.0
2,0,1,774660918,2218,-79.0,False,181,3,0,269249.0
3,0,1,774660918,1175,-91.0,False,182,4,0,8532.0
4,0,1,774660918,2339,-121.0,False,183,5,0,2341.0
5,0,1,774660918,1225,-124.0,False,184,6,0,4674.0
6,0,1,774660918,1226,-124.0,False,185,7,0,6260.0
7,0,1,774660918,1227,-124.0,False,186,8,0,1189.0
8,0,1,774660918,1228,-124.0,False,187,9,0,11204.0
9,0,1,774660918,1229,-124.0,False,188,10,0,5734.0


In [3]:
from pathlib import Path

print("=== FILES IN SELECTED CONDITION DIRECTORIES ===")

for row in selected.itertuples(index=False):
    condition_id = str(row.ConditionID)
    noise = int(row.NoisePercent)

    condition_dir = RAW_RESULT_ROOT / condition_id

    print(f"\n--- Noise {noise}% | {condition_id} ---")

    for p in sorted(condition_dir.iterdir()):
        print(
            p.name,
            "|",
            f"{p.stat().st_size:,} bytes"
        )

=== FILES IN SELECTED CONDITION DIRECTORIES ===

--- Noise 0% | noise_00__seed_01 ---
COMPLETE.json | 484 bytes
build_metrics.csv | 7,516 bytes
condition_audit.csv | 1,247 bytes
condition_summary.json | 2,828 bytes
model_fits.csv | 665 bytes
project_runs.csv | 1,301 bytes
rankings.csv.gz | 164,532 bytes
training_medians.csv | 14,390 bytes

--- Noise 5% | noise_05__seed_01 ---
COMPLETE.json | 484 bytes
build_metrics.csv | 7,514 bytes
condition_audit.csv | 1,275 bytes
condition_summary.json | 2,837 bytes
model_fits.csv | 664 bytes
project_runs.csv | 1,299 bytes
rankings.csv.gz | 173,669 bytes
training_medians.csv | 14,455 bytes


In [4]:
import pandas as pd
import json

print("=== STEP 2A — INSPECT FROZEN METRIC SCHEMA ===")

for row in selected.itertuples(index=False):

    condition_id = str(row.ConditionID)
    noise = int(row.NoisePercent)
    condition_dir = RAW_RESULT_ROOT / condition_id

    print("\n" + "=" * 80)
    print(f"NOISE {noise}% | {condition_id}")
    print("=" * 80)

    # ----------------------------------------------------------
    # 1. BUILD-LEVEL METRICS
    # ----------------------------------------------------------
    build_path = condition_dir / "build_metrics.csv"
    bm = pd.read_csv(build_path)

    print("\n--- build_metrics.csv ---")
    print("Shape:", bm.shape)
    print("Columns:")
    print(list(bm.columns))

    # Try to isolate LatestFail if technique column is identifiable
    tech_candidates = [
        c for c in bm.columns
        if c.lower() in {
            "technique",
            "method",
            "model",
            "prioritizationtechnique",
            "prioritization_technique"
        }
    ]

    if tech_candidates:
        tc = tech_candidates[0]
        lf_bm = bm[
            bm[tc].astype(str).str.lower() == "latestfail"
        ]

        print(f"\nLatestFail rows using column '{tc}':")
        display(lf_bm.head(10))
        print("LatestFail row count:", len(lf_bm))
    else:
        print("\nNo obvious technique column detected.")
        display(bm.head(10))

    # ----------------------------------------------------------
    # 2. PROJECT/RUN-LEVEL METRICS
    # ----------------------------------------------------------
    runs_path = condition_dir / "project_runs.csv"
    pr = pd.read_csv(runs_path)

    print("\n--- project_runs.csv ---")
    print("Shape:", pr.shape)
    print("Columns:")
    print(list(pr.columns))

    tech_candidates = [
        c for c in pr.columns
        if c.lower() in {
            "technique",
            "method",
            "model",
            "prioritizationtechnique",
            "prioritization_technique"
        }
    ]

    if tech_candidates:
        tc = tech_candidates[0]
        lf_pr = pr[
            pr[tc].astype(str).str.lower() == "latestfail"
        ]

        print(f"\nLatestFail rows using column '{tc}':")
        display(lf_pr)
    else:
        print("\nNo obvious technique column detected.")
        display(pr.head(10))

    # ----------------------------------------------------------
    # 3. CONDITION SUMMARY
    # ----------------------------------------------------------
    summary_path = condition_dir / "condition_summary.json"

    with open(summary_path, "r") as f:
        summary = json.load(f)

    print("\n--- condition_summary.json ---")
    print("Top-level keys:")
    print(list(summary.keys()))

    # Only print compact scalar/simple values.
    compact = {
        k: v
        for k, v in summary.items()
        if isinstance(v, (str, int, float, bool, type(None)))
    }

    print("\nSimple summary values:")
    print(compact)

=== STEP 2A — INSPECT FROZEN METRIC SCHEMA ===

NOISE 0% | noise_00__seed_01

--- build_metrics.csv ---
Shape: (56, 13)
Columns:
['ProjectNumber', 'Project', 'ProjectSlug', 'ConditionKey', 'NoisePercent', 'RepetitionSeed', 'Technique', 'Build', 'Tests', 'Failures', 'TotalDuration', 'APFDc', 'APFD']

LatestFail rows using column 'Technique':


,ProjectNumber,Project,ProjectSlug,ConditionKey,NoisePercent,RepetitionSeed,Technique,Build,Tests,Failures,TotalDuration,APFDc,APFD
40,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,LatestFail,774660918,204,1,1306241.0,0.120336,0.051471
41,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,LatestFail,774663566,204,1,1452773.0,0.387191,0.125000
42,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,LatestFail,774664908,204,2,1560296.0,0.252387,0.085784
43,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,LatestFail,774667195,204,1,1555200.0,0.423366,0.120098
44,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,LatestFail,774670605,204,1,1395521.0,0.365893,0.120098
45,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,LatestFail,774673399,202,1,1131755.0,0.232043,0.126238
46,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,LatestFail,774675402,215,1,1397817.0,0.157163,0.076744
47,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,LatestFail,774679068,204,1,1620781.0,0.458265,0.120098


LatestFail row count: 8

--- project_runs.csv ---
Shape: (7, 15)
Columns:
['ProjectNumber', 'Project', 'ProjectSlug', 'ConditionKey', 'NoisePercent', 'RepetitionSeed', 'Technique', 'EvaluationBuilds', 'ScoredFailingBuilds', 'EvaluationRows', 'EvaluationFailures', 'MeanAPFDc', 'MedianAPFDc', 'MeanAPFD', 'MedianAPFD']

LatestFail rows using column 'Technique':


,ProjectNumber,Project,ProjectSlug,ConditionKey,NoisePercent,RepetitionSeed,Technique,EvaluationBuilds,ScoredFailingBuilds,EvaluationRows,EvaluationFailures,MeanAPFDc,MedianAPFDc,MeanAPFD,MedianAPFD
5,16,apache@rocketmq,apache__rocketmq,noise_00__seed_01,0,1,LatestFail,134,8,1641,9,0.29958,0.30914,0.103191,0.120098



--- condition_summary.json ---
Top-level keys:
['AcceleratedEngineVersion', 'BaselineFingerprints', 'BuildMetricRows', 'CompletedAtUTC', 'ConditionKey', 'ConditionOrder', 'ConditionSeconds', 'DependentRECChanges', 'IndependentRECChanges', 'MLFits', 'ModelLabelChanges', 'NoisePercent', 'NumberFlipped', 'OutputManifest', 'Predictors', 'Project', 'ProjectNumber', 'ProjectRunRows', 'ProjectSlug', 'RankingRows', 'RepetitionSeed', 'Status', 'TrainingFailures', 'TrainingMedianRows']

Simple summary values:
{'AcceleratedEngineVersion': 'PROJECT_16_FAST_DEPENDENT_REC_V2_SMOKE_SCHEMA_COMPATIBLE_FROZEN_INFERRED_ORDER', 'BuildMetricRows': 56, 'CompletedAtUTC': '2026-08-06T15:18:57.275672+00:00', 'ConditionKey': 'noise_00__seed_01', 'ConditionOrder': 1, 'ConditionSeconds': 27.721837809000135, 'DependentRECChanges': 0, 'IndependentRECChanges': 0, 'MLFits': 4, 'ModelLabelChanges': 0, 'NoisePercent': 0, 'NumberFlipped': 0, 'Predictors': 151, 'Project': 'apache@rocketmq', 'ProjectNumber': 16, 'Project

,ProjectNumber,Project,ProjectSlug,ConditionKey,NoisePercent,RepetitionSeed,Technique,Build,Tests,Failures,TotalDuration,APFDc,APFD
40,16,apache@rocketmq,apache__rocketmq,noise_05__seed_01,5,1,LatestFail,774660918,204,1,1306241.0,0.034173,0.066176
41,16,apache@rocketmq,apache__rocketmq,noise_05__seed_01,5,1,LatestFail,774663566,204,1,1452773.0,0.974239,0.992647
42,16,apache@rocketmq,apache__rocketmq,noise_05__seed_01,5,1,LatestFail,774664908,204,2,1560296.0,0.727393,0.838235
43,16,apache@rocketmq,apache__rocketmq,noise_05__seed_01,5,1,LatestFail,774667195,204,1,1555200.0,0.949267,0.987745
44,16,apache@rocketmq,apache__rocketmq,noise_05__seed_01,5,1,LatestFail,774670605,204,1,1395521.0,0.965783,0.987745
45,16,apache@rocketmq,apache__rocketmq,noise_05__seed_01,5,1,LatestFail,774673399,202,1,1131755.0,0.975753,0.992574
46,16,apache@rocketmq,apache__rocketmq,noise_05__seed_01,5,1,LatestFail,774675402,215,1,1397817.0,0.427262,0.532558
47,16,apache@rocketmq,apache__rocketmq,noise_05__seed_01,5,1,LatestFail,774679068,204,1,1620781.0,0.982379,0.987745


LatestFail row count: 8

--- project_runs.csv ---
Shape: (7, 15)
Columns:
['ProjectNumber', 'Project', 'ProjectSlug', 'ConditionKey', 'NoisePercent', 'RepetitionSeed', 'Technique', 'EvaluationBuilds', 'ScoredFailingBuilds', 'EvaluationRows', 'EvaluationFailures', 'MeanAPFDc', 'MedianAPFDc', 'MeanAPFD', 'MedianAPFD']

LatestFail rows using column 'Technique':


,ProjectNumber,Project,ProjectSlug,ConditionKey,NoisePercent,RepetitionSeed,Technique,EvaluationBuilds,ScoredFailingBuilds,EvaluationRows,EvaluationFailures,MeanAPFDc,MedianAPFDc,MeanAPFD,MedianAPFD
5,16,apache@rocketmq,apache__rocketmq,noise_05__seed_01,5,1,LatestFail,134,8,1641,9,0.754531,0.957525,0.798178,0.987745



--- condition_summary.json ---
Top-level keys:
['AcceleratedEngineVersion', 'BaselineFingerprints', 'BuildMetricRows', 'CompletedAtUTC', 'ConditionKey', 'ConditionOrder', 'ConditionSeconds', 'DependentRECChanges', 'IndependentRECChanges', 'MLFits', 'ModelLabelChanges', 'NoisePercent', 'NumberFlipped', 'OutputManifest', 'Predictors', 'Project', 'ProjectNumber', 'ProjectRunRows', 'ProjectSlug', 'RankingRows', 'RepetitionSeed', 'Status', 'TrainingFailures', 'TrainingMedianRows']

Simple summary values:
{'AcceleratedEngineVersion': 'PROJECT_16_FAST_DEPENDENT_REC_V2_SMOKE_SCHEMA_COMPATIBLE_FROZEN_INFERRED_ORDER', 'BuildMetricRows': 56, 'CompletedAtUTC': '2026-08-06T15:19:48.584503+00:00', 'ConditionKey': 'noise_05__seed_01', 'ConditionOrder': 2, 'ConditionSeconds': 14.988177791999988, 'DependentRECChanges': 54760, 'IndependentRECChanges': 0, 'MLFits': 4, 'ModelLabelChanges': 241, 'NoisePercent': 5, 'NumberFlipped': 3581, 'Predictors': 151, 'Project': 'apache@rocketmq', 'ProjectNumber': 16,

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
from datetime import datetime, timezone

# ================================================================
# STEP 3 — GLOBAL LATESTFAIL SENTINEL CORRECTION
#
# READS:
#   frozen Results/Raw/<project>/<condition>/rankings.csv.gz
#
# WRITES:
#   Results/Corrections/LatestFail_Sentinel_Fix_v1/
#
# DOES NOT modify any frozen experiment output.
# ================================================================

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
RESULTS_ROOT = THESIS_ROOT / "Results"
RAW_ROOT = RESULTS_ROOT / "Raw"

CORRECTION_ROOT = (
    RESULTS_ROOT
    / "Corrections"
    / "LatestFail_Sentinel_Fix_v1"
)

PROJECT_OUTPUT_ROOT = CORRECTION_ROOT / "projects"
VALIDATION_ROOT = CORRECTION_ROOT / "validation"

PROJECT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_NOISE_LEVELS = {
    0, 5, 10, 15, 20, 25, 30, 40, 50
}
EXPECTED_SEEDS = set(range(1, 31))
EXPECTED_CONDITIONS_PER_PROJECT = 270

SENTINEL_SCORE = 1.0
TOL = 1e-12


# ---------------------------------------------------------------
# Metric functions already validated against frozen project_runs
# ---------------------------------------------------------------

def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)

    n = len(failures)
    m = int(failures.sum())

    if n == 0 or m == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - failure_positions.sum() / (n * m)
        + 1.0 / (2.0 * n)
    )


def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    n = len(failures)
    m = int(failures.sum())

    if n == 0 or m == 0:
        return np.nan

    total_duration = durations.sum()

    if not np.isfinite(total_duration) or total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate(
        [
            np.array([0.0]),
            np.cumsum(durations)[:-1],
        ]
    )

    failure_mask = failures == 1

    midpoint_times = (
        cumulative_before[failure_mask]
        + 0.5 * durations[failure_mask]
    )

    return float(
        1.0
        - np.mean(midpoint_times / total_duration)
    )


def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ---------------------------------------------------------------
# Discover projects containing COMPLETE frozen conditions
# ---------------------------------------------------------------

project_dirs = sorted(
    p for p in RAW_ROOT.iterdir()
    if p.is_dir()
)

print("Raw project directories discovered:", len(project_dirs))


global_build_metric_frames = []
global_project_run_rows = []
condition_validation_rows = []
project_summary_rows = []

processed_projects = 0
processed_conditions = 0


for project_dir in project_dirs:

    project_slug = project_dir.name

    condition_dirs = sorted(
        p for p in project_dir.iterdir()
        if (
            p.is_dir()
            and p.name.startswith("noise_")
            and (p / "COMPLETE.json").is_file()
            and (p / "rankings.csv.gz").is_file()
            and (p / "project_runs.csv").is_file()
        )
    )

    # Ignore directories that are not complete 270-condition studies.
    if len(condition_dirs) != EXPECTED_CONDITIONS_PER_PROJECT:
        print(
            f"SKIP {project_slug}: "
            f"{len(condition_dirs)} completed conditions "
            f"(expected 270)"
        )
        continue

    # -----------------------------------------------------------
    # Validate project condition grid before processing
    # -----------------------------------------------------------

    observed_pairs = set()

    for condition_dir in condition_dirs:
        pr = pd.read_csv(
            condition_dir / "project_runs.csv"
        )

        lf = pr[
            pr["Technique"].astype(str) == "LatestFail"
        ]

        if len(lf) != 1:
            raise RuntimeError(
                f"{project_slug}/{condition_dir.name}: "
                f"expected exactly one LatestFail project_runs row, "
                f"found {len(lf)}"
            )

        observed_pairs.add(
            (
                int(lf.iloc[0]["NoisePercent"]),
                int(lf.iloc[0]["RepetitionSeed"]),
            )
        )

    expected_pairs = {
        (noise, seed)
        for noise in EXPECTED_NOISE_LEVELS
        for seed in EXPECTED_SEEDS
    }

    if observed_pairs != expected_pairs:
        missing = sorted(expected_pairs - observed_pairs)
        unexpected = sorted(observed_pairs - expected_pairs)

        raise RuntimeError(
            f"{project_slug}: invalid condition grid.\n"
            f"Missing: {missing[:20]}\n"
            f"Unexpected: {unexpected[:20]}"
        )

    project_build_frames = []
    project_run_rows = []

    project_sentinel_total = 0
    project_latestfail_rows_total = 0
    project_changed_conditions = 0

    # -----------------------------------------------------------
    # Process all 270 conditions
    # -----------------------------------------------------------

    for condition_dir in condition_dirs:

        rankings_path = (
            condition_dir / "rankings.csv.gz"
        )

        frozen_runs_path = (
            condition_dir / "project_runs.csv"
        )

        rankings = pd.read_csv(rankings_path)

        required_cols = {
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Score",
            "Rank",
            "CleanFailure",
            "Duration",
        }

        missing_cols = (
            required_cols - set(rankings.columns)
        )

        if missing_cols:
            raise RuntimeError(
                f"{project_slug}/{condition_dir.name}: "
                f"missing ranking columns {missing_cols}"
            )

        lf = rankings[
            rankings["Technique"].astype(str)
            == "LatestFail"
        ].copy()

        if lf.empty:
            raise RuntimeError(
                f"{project_slug}/{condition_dir.name}: "
                "no LatestFail ranking rows"
            )

        # Metadata must be unique within one condition.
        metadata_cols = [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
        ]

        metadata = {}

        for col in metadata_cols:
            values = lf[col].drop_duplicates()

            if len(values) != 1:
                raise RuntimeError(
                    f"{project_slug}/{condition_dir.name}: "
                    f"non-unique {col}: {values.tolist()}"
                )

            metadata[col] = values.iloc[0]

        noise = int(metadata["NoisePercent"])
        seed = int(metadata["RepetitionSeed"])

        # -------------------------------------------------------
        # Sentinel identification
        #
        # Original implementation:
        #   Score = -REC_LastFailureAge
        #
        # Therefore:
        #   REC_LastFailureAge = -1
        #   -> Score = +1
        # -------------------------------------------------------

        lf["NeverFailed"] = np.isclose(
            lf["Score"].to_numpy(dtype=float),
            SENTINEL_SCORE,
            rtol=0,
            atol=TOL,
        )

        sentinel_rows = int(
            lf["NeverFailed"].sum()
        )

        latestfail_rows = len(lf)

        project_sentinel_total += sentinel_rows
        project_latestfail_rows_total += latestfail_rows

        # -------------------------------------------------------
        # Validate old ranking against frozen project_runs.csv
        # BEFORE changing anything.
        # -------------------------------------------------------

        frozen_runs = pd.read_csv(
            frozen_runs_path
        )

        frozen_lf = frozen_runs[
            frozen_runs["Technique"].astype(str)
            == "LatestFail"
        ].copy()

        if len(frozen_lf) != 1:
            raise RuntimeError(
                f"{project_slug}/{condition_dir.name}: "
                "invalid frozen LatestFail project_runs row count"
            )

        old_build_rows = []

        for build, g in lf.groupby(
            "Build",
            sort=False
        ):
            g = g.sort_values(
                "Rank",
                kind="mergesort"
            )

            failures = int(
                g["CleanFailure"].sum()
            )

            if failures == 0:
                continue

            old_build_rows.append(
                {
                    "APFD": calculate_apfd(
                        g["CleanFailure"]
                    ),
                    "APFDc": calculate_apfdc(
                        g["CleanFailure"],
                        g["Duration"],
                    ),
                }
            )

        old_metric_df = pd.DataFrame(
            old_build_rows
        )

        reconstructed_old_apfd = float(
            old_metric_df["APFD"].mean()
        )

        reconstructed_old_apfdc = float(
            old_metric_df["APFDc"].mean()
        )

        frozen_old_apfd = float(
            frozen_lf.iloc[0]["MeanAPFD"]
        )

        frozen_old_apfdc = float(
            frozen_lf.iloc[0]["MeanAPFDc"]
        )

        old_apfd_match = np.isclose(
            reconstructed_old_apfd,
            frozen_old_apfd,
            rtol=1e-10,
            atol=1e-12,
        )

        old_apfdc_match = np.isclose(
            reconstructed_old_apfdc,
            frozen_old_apfdc,
            rtol=1e-10,
            atol=1e-12,
        )

        if not (
            old_apfd_match
            and old_apfdc_match
        ):
            raise RuntimeError(
                f"{project_slug}/{condition_dir.name}: "
                "reconstructed old metrics do not match "
                "frozen project_runs.\n"
                f"APFD: reconstructed={reconstructed_old_apfd}, "
                f"frozen={frozen_old_apfd}\n"
                f"APFDc: reconstructed={reconstructed_old_apfdc}, "
                f"frozen={frozen_old_apfdc}"
            )

        # -------------------------------------------------------
        # CORRECTED LATESTFAIL ORDER
        #
        # 1. Tests with an actual previous failure first.
        # 2. Within those tests, higher old Score first
        #    (= smaller REC_LastFailureAge).
        # 3. Never-failed tests last.
        # 4. Test identifier preserves deterministic tie-break.
        # -------------------------------------------------------

        corrected = (
            lf.sort_values(
                [
                    "Build",
                    "NeverFailed",
                    "Score",
                    "Test",
                ],
                ascending=[
                    True,
                    True,
                    False,
                    True,
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        corrected["CorrectedRank"] = (
            corrected
            .groupby("Build", sort=False)
            .cumcount()
            .add(1)
        )

        # -------------------------------------------------------
        # Strong ordering validation:
        # among non-sentinel tests, relative score ordering
        # must remain descending.
        # -------------------------------------------------------

        for build, g in corrected.groupby(
            "Build",
            sort=False
        ):
            actual_failure_history = g[
                ~g["NeverFailed"]
            ]

            scores = (
                actual_failure_history["Score"]
                .to_numpy(dtype=float)
            )

            if (
                len(scores) > 1
                and np.any(
                    scores[:-1]
                    < scores[1:] - TOL
                )
            ):
                raise RuntimeError(
                    f"{project_slug}/{condition_dir.name}/"
                    f"build={build}: "
                    "non-sentinel LatestFail score order changed"
                )

            # No never-failed row may precede an actual-history row.
            flags = (
                g["NeverFailed"]
                .to_numpy(dtype=bool)
            )

            if np.any(
                flags[:-1] & (~flags[1:])
            ):
                raise RuntimeError(
                    f"{project_slug}/{condition_dir.name}/"
                    f"build={build}: sentinel ordering invalid"
                )

        # -------------------------------------------------------
        # Corrected build metrics
        # -------------------------------------------------------

        corrected_build_rows = []

        for build, g in corrected.groupby(
            "Build",
            sort=False
        ):

            tests = len(g)
            failures = int(
                g["CleanFailure"].sum()
            )

            total_duration = float(
                g["Duration"].sum()
            )

            # Frozen build_metrics stores only failing builds.
            if failures == 0:
                continue

            apfd = calculate_apfd(
                g["CleanFailure"]
            )

            apfdc = calculate_apfdc(
                g["CleanFailure"],
                g["Duration"],
            )

            corrected_build_rows.append(
                {
                    "ProjectNumber":
                        metadata["ProjectNumber"],
                    "Project":
                        metadata["Project"],
                    "ProjectSlug":
                        metadata["ProjectSlug"],
                    "ConditionKey":
                        metadata["ConditionKey"],
                    "NoisePercent":
                        noise,
                    "RepetitionSeed":
                        seed,
                    "Technique":
                        "LatestFail",
                    "Build":
                        build,
                    "Tests":
                        tests,
                    "Failures":
                        failures,
                    "TotalDuration":
                        total_duration,
                    "APFDc":
                        apfdc,
                    "APFD":
                        apfd,
                }
            )

        corrected_build_df = pd.DataFrame(
            corrected_build_rows
        )

        if corrected_build_df.empty:
            raise RuntimeError(
                f"{project_slug}/{condition_dir.name}: "
                "no failing evaluation builds"
            )

        # -------------------------------------------------------
        # Corrected project-run summary
        # -------------------------------------------------------

        evaluation_builds = int(
            lf["Build"].nunique()
        )

        scored_failing_builds = int(
            len(corrected_build_df)
        )

        evaluation_rows = int(
            len(lf)
        )

        evaluation_failures = int(
            lf["CleanFailure"].sum()
        )

        corrected_mean_apfdc = float(
            corrected_build_df["APFDc"].mean()
        )

        corrected_median_apfdc = float(
            corrected_build_df["APFDc"].median()
        )

        corrected_mean_apfd = float(
            corrected_build_df["APFD"].mean()
        )

        corrected_median_apfd = float(
            corrected_build_df["APFD"].median()
        )

        project_run_row = {
            "ProjectNumber":
                metadata["ProjectNumber"],
            "Project":
                metadata["Project"],
            "ProjectSlug":
                metadata["ProjectSlug"],
            "ConditionKey":
                metadata["ConditionKey"],
            "NoisePercent":
                noise,
            "RepetitionSeed":
                seed,
            "Technique":
                "LatestFail",
            "EvaluationBuilds":
                evaluation_builds,
            "ScoredFailingBuilds":
                scored_failing_builds,
            "EvaluationRows":
                evaluation_rows,
            "EvaluationFailures":
                evaluation_failures,
            "MeanAPFDc":
                corrected_mean_apfdc,
            "MedianAPFDc":
                corrected_median_apfdc,
            "MeanAPFD":
                corrected_mean_apfd,
            "MedianAPFD":
                corrected_median_apfd,
        }

        old_vs_new_changed = (
            not np.isclose(
                corrected_mean_apfd,
                frozen_old_apfd,
                rtol=1e-12,
                atol=1e-12,
            )
            or
            not np.isclose(
                corrected_mean_apfdc,
                frozen_old_apfdc,
                rtol=1e-12,
                atol=1e-12,
            )
        )

        if old_vs_new_changed:
            project_changed_conditions += 1

        # -------------------------------------------------------
        # Validation record
        # -------------------------------------------------------

        condition_validation_rows.append(
            {
                "ProjectNumber":
                    metadata["ProjectNumber"],
                "Project":
                    metadata["Project"],
                "ProjectSlug":
                    metadata["ProjectSlug"],
                "ConditionKey":
                    metadata["ConditionKey"],
                "NoisePercent":
                    noise,
                "RepetitionSeed":
                    seed,

                "LatestFailRows":
                    latestfail_rows,

                "SentinelRows":
                    sentinel_rows,

                "SentinelPercent":
                    100.0
                    * sentinel_rows
                    / latestfail_rows,

                "FrozenMeanAPFD":
                    frozen_old_apfd,

                "ReconstructedOldMeanAPFD":
                    reconstructed_old_apfd,

                "OldAPFDMatch":
                    old_apfd_match,

                "CorrectedMeanAPFD":
                    corrected_mean_apfd,

                "APFDChange":
                    corrected_mean_apfd
                    - frozen_old_apfd,

                "FrozenMeanAPFDc":
                    frozen_old_apfdc,

                "ReconstructedOldMeanAPFDc":
                    reconstructed_old_apfdc,

                "OldAPFDcMatch":
                    old_apfdc_match,

                "CorrectedMeanAPFDc":
                    corrected_mean_apfdc,

                "APFDcChange":
                    corrected_mean_apfdc
                    - frozen_old_apfdc,

                "ConditionChanged":
                    old_vs_new_changed,

                "SourceRankingsSHA256":
                    sha256_file(
                        rankings_path
                    ),
            }
        )

        project_build_frames.append(
            corrected_build_df
        )

        project_run_rows.append(
            project_run_row
        )

        processed_conditions += 1

    # -----------------------------------------------------------
    # Save corrected project outputs
    # -----------------------------------------------------------

    project_out = (
        PROJECT_OUTPUT_ROOT
        / project_slug
    )

    project_out.mkdir(
        parents=True,
        exist_ok=True
    )

    project_build_df = pd.concat(
        project_build_frames,
        ignore_index=True,
    )

    project_runs_df = pd.DataFrame(
        project_run_rows
    ).sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
        ]
    ).reset_index(drop=True)

    if len(project_runs_df) != 270:
        raise RuntimeError(
            f"{project_slug}: expected 270 corrected "
            f"project-run rows, got {len(project_runs_df)}"
        )

    project_build_path = (
        project_out
        / "LatestFail_corrected_build_metrics.csv"
    )

    project_runs_path = (
        project_out
        / "LatestFail_corrected_project_runs.csv"
    )

    project_build_df.to_csv(
        project_build_path,
        index=False,
    )

    project_runs_df.to_csv(
        project_runs_path,
        index=False,
    )

    global_build_metric_frames.append(
        project_build_df
    )

    global_project_run_rows.extend(
        project_run_rows
    )

    project_summary_rows.append(
        {
            "ProjectSlug":
                project_slug,
            "Conditions":
                len(project_runs_df),
            "LatestFailRows":
                project_latestfail_rows_total,
            "SentinelRows":
                project_sentinel_total,
            "SentinelPercent":
                (
                    100.0
                    * project_sentinel_total
                    / project_latestfail_rows_total
                ),
            "ChangedConditions":
                project_changed_conditions,
            "CorrectedProjectRunsSHA256":
                sha256_file(
                    project_runs_path
                ),
            "CorrectedBuildMetricsSHA256":
                sha256_file(
                    project_build_path
                ),
        }
    )

    processed_projects += 1

    print(
        f"PASS {project_slug}: "
        f"270 conditions | "
        f"sentinel={project_sentinel_total:,}/"
        f"{project_latestfail_rows_total:,} "
        f"({100*project_sentinel_total/project_latestfail_rows_total:.2f}%) | "
        f"changed={project_changed_conditions}/270"
    )


# ================================================================
# GLOBAL OUTPUTS
# ================================================================

if processed_projects == 0:
    raise RuntimeError(
        "No complete 270-condition projects were processed."
    )

global_project_runs = pd.DataFrame(
    global_project_run_rows
).sort_values(
    [
        "ProjectNumber",
        "NoisePercent",
        "RepetitionSeed",
    ]
).reset_index(drop=True)

global_build_metrics = pd.concat(
    global_build_metric_frames,
    ignore_index=True,
)

validation_df = pd.DataFrame(
    condition_validation_rows
).sort_values(
    [
        "ProjectNumber",
        "NoisePercent",
        "RepetitionSeed",
    ]
).reset_index(drop=True)

project_summary_df = pd.DataFrame(
    project_summary_rows
).sort_values(
    "ProjectSlug"
).reset_index(drop=True)


# ---------------------------------------------------------------
# Fail-fast global validation
# ---------------------------------------------------------------

if not validation_df["OldAPFDMatch"].all():
    raise RuntimeError(
        "At least one old APFD reconstruction did not "
        "match the frozen experiment."
    )

if not validation_df["OldAPFDcMatch"].all():
    raise RuntimeError(
        "At least one old APFDc reconstruction did not "
        "match the frozen experiment."
    )

expected_global_rows = (
    processed_projects
    * EXPECTED_CONDITIONS_PER_PROJECT
)

if len(global_project_runs) != expected_global_rows:
    raise RuntimeError(
        f"Expected {expected_global_rows} corrected "
        f"project-run rows, found "
        f"{len(global_project_runs)}"
    )


# ---------------------------------------------------------------
# Save global artifacts
# ---------------------------------------------------------------

global_runs_path = (
    CORRECTION_ROOT
    / "LatestFail_corrected_project_runs_ALL.csv"
)

global_build_path = (
    CORRECTION_ROOT
    / "LatestFail_corrected_build_metrics_ALL.csv"
)

validation_path = (
    VALIDATION_ROOT
    / "LatestFail_condition_validation.csv"
)

project_summary_path = (
    VALIDATION_ROOT
    / "LatestFail_project_summary.csv"
)

global_project_runs.to_csv(
    global_runs_path,
    index=False,
)

global_build_metrics.to_csv(
    global_build_path,
    index=False,
)

validation_df.to_csv(
    validation_path,
    index=False,
)

project_summary_df.to_csv(
    project_summary_path,
    index=False,
)


# ---------------------------------------------------------------
# Global summary by noise level
#
# IMPORTANT:
# Equal project weighting:
# first average 30 seeds within each project,
# then average the projects.
# ---------------------------------------------------------------

project_noise = (
    global_project_runs
    .groupby(
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "NoisePercent",
        ],
        as_index=False,
    )
    .agg(
        ProjectMeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),
        ProjectMeanAPFD=(
            "MeanAPFD",
            "mean",
        ),
    )
)

global_noise_summary = (
    project_noise
    .groupby(
        "NoisePercent",
        as_index=False,
    )
    .agg(
        Projects=(
            "ProjectSlug",
            "nunique",
        ),
        CorrectedLatestFailMeanAPFDc=(
            "ProjectMeanAPFDc",
            "mean",
        ),
        CorrectedLatestFailMeanAPFD=(
            "ProjectMeanAPFD",
            "mean",
        ),
        MedianProjectAPFDc=(
            "ProjectMeanAPFDc",
            "median",
        ),
        MedianProjectAPFD=(
            "ProjectMeanAPFD",
            "median",
        ),
    )
    .sort_values(
        "NoisePercent"
    )
    .reset_index(drop=True)
)

global_summary_path = (
    CORRECTION_ROOT
    / "LatestFail_corrected_global_noise_summary.csv"
)

global_noise_summary.to_csv(
    global_summary_path,
    index=False,
)


# ---------------------------------------------------------------
# Correction manifest
# ---------------------------------------------------------------

manifest = {
    "CorrectionID":
        "LatestFail_Sentinel_Fix_v1",

    "CreatedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "SourceRoot":
        str(RAW_ROOT),

    "SourceModificationPolicy":
        "READ_ONLY",

    "Correction":
        (
            "REC_LastFailureAge=-1 means no prior failure. "
            "The frozen implementation used "
            "Score=-REC_LastFailureAge and descending ranking, "
            "which assigned sentinel rows Score=+1 and placed "
            "them ahead of tests with actual prior failures. "
            "The correction places sentinel rows after all tests "
            "with a recorded prior failure while preserving "
            "descending score order and deterministic Test "
            "tie-breaking among all other rows."
        ),

    "ProcessedProjects":
        int(processed_projects),

    "ProcessedConditions":
        int(processed_conditions),

    "CorrectedProjectRunRows":
        int(len(global_project_runs)),

    "ValidationAllOldAPFDMatch":
        bool(
            validation_df[
                "OldAPFDMatch"
            ].all()
        ),

    "ValidationAllOldAPFDcMatch":
        bool(
            validation_df[
                "OldAPFDcMatch"
            ].all()
        ),

    "GlobalCorrectedProjectRunsSHA256":
        sha256_file(
            global_runs_path
        ),

    "GlobalCorrectedBuildMetricsSHA256":
        sha256_file(
            global_build_path
        ),

    "ValidationSHA256":
        sha256_file(
            validation_path
        ),

    "GlobalNoiseSummarySHA256":
        sha256_file(
            global_summary_path
        ),
}

manifest_path = (
    CORRECTION_ROOT
    / "CORRECTION_MANIFEST.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        sort_keys=True,
    )


print("\n" + "=" * 80)
print("LATESTFAIL GLOBAL CORRECTION COMPLETE")
print("=" * 80)

print(
    "Projects processed:",
    processed_projects
)

print(
    "Conditions processed:",
    processed_conditions
)

print(
    "Corrected project-run rows:",
    len(global_project_runs)
)

print(
    "All old APFD reconstructions match frozen:",
    validation_df["OldAPFDMatch"].all()
)

print(
    "All old APFDc reconstructions match frozen:",
    validation_df["OldAPFDcMatch"].all()
)

print("\n=== CORRECTED GLOBAL LATESTFAIL MEANS ===")
display(global_noise_summary)

print("\n=== PROJECT-LEVEL CORRECTION SUMMARY ===")
display(project_summary_df)

print("\nCorrection root:")
print(CORRECTION_ROOT)

print("\nManifest:")
print(manifest_path)

Raw project directories discovered: 24
SKIP Angel-ML__angel: 0 completed conditions (expected 270)
SKIP CompEvol__beast2: 0 completed conditions (expected 270)


KeyboardInterrupt: 

In [6]:
from pathlib import Path

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
RAW_ROOT = THESIS_ROOT / "Results" / "Raw"

print("=== RAW PROJECT STRUCTURE AUDIT ===\n")

project_dirs = sorted(
    p for p in RAW_ROOT.iterdir()
    if p.is_dir()
)

print("Project directories:", len(project_dirs))
print()

for project_dir in project_dirs:

    direct_conditions = [
        p for p in project_dir.iterdir()
        if p.is_dir() and p.name.startswith("noise_")
    ]

    complete_direct = [
        p for p in direct_conditions
        if (p / "COMPLETE.json").is_file()
    ]

    print("=" * 80)
    print("PROJECT:", project_dir.name)
    print("PATH:", project_dir)
    print("Direct noise dirs:", len(direct_conditions))
    print("Direct COMPLETE conditions:", len(complete_direct))

    if len(complete_direct) == 0:
        print("Top-level contents:")
        for x in list(sorted(project_dir.iterdir()))[:20]:
            print("   ", x.name, "[DIR]" if x.is_dir() else "[FILE]")

print("\n=== END STRUCTURE AUDIT ===")

=== RAW PROJECT STRUCTURE AUDIT ===

Project directories: 24

PROJECT: Angel-ML__angel
PATH: /content/drive/MyDrive/Thesis_Experiment/Results/Raw/Angel-ML__angel
Direct noise dirs: 9
Direct COMPLETE conditions: 0
Top-level contents:
    noise_000 [DIR]
    noise_005 [DIR]
    noise_010 [DIR]
    noise_015 [DIR]
    noise_020 [DIR]
    noise_025 [DIR]
    noise_030 [DIR]
    noise_040 [DIR]
    noise_050 [DIR]
PROJECT: CompEvol__beast2
PATH: /content/drive/MyDrive/Thesis_Experiment/Results/Raw/CompEvol__beast2
Direct noise dirs: 0
Direct COMPLETE conditions: 0
Top-level contents:
    beast2_30_seed_raw [DIR]
PROJECT: EMResearch__EvoMaster
PATH: /content/drive/MyDrive/Thesis_Experiment/Results/Raw/EMResearch__EvoMaster
Direct noise dirs: 270
Direct COMPLETE conditions: 270
PROJECT: JMRI__JMRI
PATH: /content/drive/MyDrive/Thesis_Experiment/Results/Raw/JMRI__JMRI
Direct noise dirs: 270
Direct COMPLETE conditions: 270
PROJECT: SonarSource__sonarqube
PATH: /content/drive/MyDrive/Thesis_Exper

In [7]:
from pathlib import Path
import os
from collections import Counter

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
RAW_ROOT = THESIS_ROOT / "Results" / "Raw"

print("=== UNIVERSAL CONDITION DISCOVERY AUDIT ===\n")

project_dirs = sorted(
    p for p in RAW_ROOT.iterdir()
    if p.is_dir()
)

all_discovered = {}

for project_dir in project_dirs:

    condition_dirs = []

    # Walk the project's directory tree without opening any files.
    for root, dirs, files in os.walk(project_dir):

        file_set = set(files)

        if (
            "rankings.csv.gz" in file_set
            and "project_runs.csv" in file_set
        ):
            condition_dirs.append(Path(root))

    condition_dirs = sorted(condition_dirs)
    all_discovered[project_dir.name] = condition_dirs

    # Relative-depth information helps us understand old/new layouts.
    depth_counts = Counter(
        len(p.relative_to(project_dir).parts)
        for p in condition_dirs
    )

    print("=" * 90)
    print("PROJECT:", project_dir.name)
    print("Discovered conditions:", len(condition_dirs))
    print("Relative depth distribution:", dict(sorted(depth_counts.items())))

    if condition_dirs:
        print("First 3:")
        for p in condition_dirs[:3]:
            print("   ", p.relative_to(project_dir))

        print("Last 3:")
        for p in condition_dirs[-3:]:
            print("   ", p.relative_to(project_dir))

print("\n" + "=" * 90)

counts = {
    project: len(paths)
    for project, paths in all_discovered.items()
}

print("PROJECT COUNT:", len(counts))
print("CONDITION COUNTS:", sorted(set(counts.values())))

bad = {
    project: count
    for project, count in counts.items()
    if count != 270
}

print("\nProjects not equal to 270:")
print(bad if bad else "NONE")

print(
    "\nTOTAL CONDITIONS:",
    sum(counts.values())
)

print("=== END UNIVERSAL DISCOVERY AUDIT ===")

=== UNIVERSAL CONDITION DISCOVERY AUDIT ===

PROJECT: Angel-ML__angel
Discovered conditions: 0
Relative depth distribution: {}
PROJECT: CompEvol__beast2
Discovered conditions: 0
Relative depth distribution: {}
PROJECT: EMResearch__EvoMaster
Discovered conditions: 270
Relative depth distribution: {1: 270}
First 3:
    noise_00__seed_01
    noise_00__seed_02
    noise_00__seed_03
Last 3:
    noise_50__seed_28
    noise_50__seed_29
    noise_50__seed_30
PROJECT: JMRI__JMRI
Discovered conditions: 270
Relative depth distribution: {1: 270}
First 3:
    noise_00__seed_01
    noise_00__seed_02
    noise_00__seed_03
Last 3:
    noise_50__seed_28
    noise_50__seed_29
    noise_50__seed_30
PROJECT: SonarSource__sonarqube
Discovered conditions: 270
Relative depth distribution: {1: 270}
First 3:
    noise_00__seed_01
    noise_00__seed_02
    noise_00__seed_03
Last 3:
    noise_50__seed_28
    noise_50__seed_29
    noise_50__seed_30
PROJECT: apache__airavata
Discovered conditions: 0
Relative depth

In [8]:
from pathlib import Path
import os
from collections import Counter, defaultdict

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
RAW_ROOT = THESIS_ROOT / "Results" / "Raw"

UNRESOLVED = [
    "Angel-ML__angel",
    "CompEvol__beast2",
    "apache__airavata",
    "b2ihealthcare__snow-owl",
    "camunda__camunda-bpm-platform",
    "eclipse__jetty.project",
    "eclipse__paho.mqtt.java",
    "optimatika__ojAlgo",
    "spring-cloud__spring-cloud-dataflow",
    "thinkaurelius__titan",
]

KEYWORDS = (
    "rank",
    "project_run",
    "project-run",
    "build_metric",
    "build-metric",
    "metric",
    "condition",
    "complete",
    "summary",
)

print("=== LEGACY PROJECT ARTIFACT INVENTORY ===")

for slug in UNRESOLVED:

    root = RAW_ROOT / slug

    print("\n" + "=" * 100)
    print("PROJECT:", slug)
    print("ROOT:", root)

    if not root.is_dir():
        print("ERROR: project directory missing")
        continue

    total_dirs = 0
    total_files = 0

    extension_counts = Counter()
    basename_counts = Counter()
    candidate_paths = []
    shallow_examples = defaultdict(list)

    for current_root, dirs, files in os.walk(root):

        total_dirs += len(dirs)
        total_files += len(files)

        current_path = Path(current_root)
        rel_dir = current_path.relative_to(root)
        depth = len(rel_dir.parts)

        # Keep a few shallow directory examples.
        if depth <= 2:
            for d in dirs[:10]:
                if len(shallow_examples[f"dirs_depth_{depth}"]) < 20:
                    shallow_examples[f"dirs_depth_{depth}"].append(
                        str((rel_dir / d))
                    )

        for filename in files:

            path = current_path / filename
            rel = path.relative_to(root)

            lower = filename.lower()

            basename_counts[filename] += 1

            # Track useful suffix combinations.
            if lower.endswith(".csv.gz"):
                extension_counts[".csv.gz"] += 1
            elif lower.endswith(".json.gz"):
                extension_counts[".json.gz"] += 1
            elif lower.endswith(".parquet"):
                extension_counts[".parquet"] += 1
            else:
                suffix = Path(filename).suffix.lower() or "<no suffix>"
                extension_counts[suffix] += 1

            if any(k in lower for k in KEYWORDS):
                if len(candidate_paths) < 80:
                    candidate_paths.append(str(rel))

    print("Total subdirectories:", total_dirs)
    print("Total files:", total_files)

    print("\nMost common file types:")
    for ext, count in extension_counts.most_common(15):
        print(f"  {ext:12s} {count}")

    print("\nMost common exact filenames:")
    for name, count in basename_counts.most_common(20):
        print(f"  {count:5d}  {name}")

    print("\nShallow directory examples:")
    for key in sorted(shallow_examples):
        print(f"  {key}:")
        for x in shallow_examples[key][:10]:
            print("     ", x)

    print("\nCandidate ranking/metric/condition files:")
    if candidate_paths:
        for p in candidate_paths[:80]:
            print("   ", p)
    else:
        print("   NONE FOUND")

print("\n=== END LEGACY PROJECT ARTIFACT INVENTORY ===")

=== LEGACY PROJECT ARTIFACT INVENTORY ===

PROJECT: Angel-ML__angel
ROOT: /content/drive/MyDrive/Thesis_Experiment/Results/Raw/Angel-ML__angel
Total subdirectories: 279
Total files: 1890

Most common file types:
  .parquet     810
  .csv         810
  .json        270

Most common exact filenames:
    270  noise_manifest.parquet
    270  noise_summary.csv
    270  build_metrics.parquet
    270  project_run_metrics.csv
    270  fit_times.csv
    270  rankings.parquet
    270  _SUCCESS.json

Shallow directory examples:
  dirs_depth_0:
      noise_000
      noise_005
      noise_010
      noise_015
      noise_020
      noise_025
      noise_030
      noise_040
      noise_050
  dirs_depth_1:
      noise_000/seed_01
      noise_000/seed_02
      noise_000/seed_03
      noise_000/seed_04
      noise_000/seed_05
      noise_000/seed_06
      noise_000/seed_07
      noise_000/seed_08
      noise_000/seed_09
      noise_000/seed_10

Candidate ranking/metric/condition files:
    noise_000/seed

In [9]:
from pathlib import Path
import os
import pandas as pd
from collections import Counter, defaultdict

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
RAW_ROOT = THESIS_ROOT / "Results" / "Raw"

RANKING_FILES = [
    "rankings.csv.gz",
    "rankings.parquet",
]

PROJECT_RUN_FILES = [
    "project_runs.csv",
    "project_run.csv",
    "project_run_metrics.csv",
]

# ================================================================
# STEP 3C — UNIVERSAL ARTIFACT ADAPTER AUDIT
# ================================================================

def detect_condition_artifacts(directory):
    files = {p.name: p for p in directory.iterdir() if p.is_file()}

    ranking_path = None
    run_path = None

    for name in RANKING_FILES:
        if name in files:
            ranking_path = files[name]
            break

    for name in PROJECT_RUN_FILES:
        if name in files:
            run_path = files[name]
            break

    if ranking_path is not None and run_path is not None:
        return ranking_path, run_path

    return None


all_conditions = []
format_counter = Counter()
project_counts = Counter()
project_formats = defaultdict(Counter)

project_dirs = sorted(
    p for p in RAW_ROOT.iterdir()
    if p.is_dir()
)

print("=== UNIVERSAL SOURCE ADAPTER DISCOVERY ===\n")

for project_root in project_dirs:

    for current_root, dirs, files in os.walk(project_root):
        current_dir = Path(current_root)

        detected = detect_condition_artifacts(current_dir)

        if detected is None:
            continue

        ranking_path, run_path = detected

        fmt = (
            ranking_path.name,
            run_path.name,
        )

        record = {
            "ProjectSlug": project_root.name,
            "ConditionDir": current_dir,
            "RankingPath": ranking_path,
            "RunPath": run_path,
            "Format": fmt,
        }

        all_conditions.append(record)
        project_counts[project_root.name] += 1
        format_counter[fmt] += 1
        project_formats[project_root.name][fmt] += 1


print("Projects discovered:", len(project_counts))
print("Total conditions discovered:", len(all_conditions))
print()

print("=== FORMAT COUNTS ===")
for fmt, count in sorted(format_counter.items()):
    print(
        f"{count:5d} conditions | "
        f"{fmt[0]} + {fmt[1]}"
    )

print("\n=== PROJECT COUNTS ===")

bad = {}

for project in sorted(project_counts):
    count = project_counts[project]

    print(f"{project:45s} {count:3d}")

    if count != 270:
        bad[project] = count

print("\nProjects not equal to 270:")
print(bad if bad else "NONE")

print("\nExpected total:", 24 * 270)
print("Observed total:", len(all_conditions))


# ================================================================
# REPRESENTATIVE SCHEMA INSPECTION
# One condition from each detected storage format
# ================================================================

print("\n\n=== REPRESENTATIVE FORMAT SCHEMA AUDIT ===")

representatives = {}

for rec in all_conditions:
    if rec["Format"] not in representatives:
        representatives[rec["Format"]] = rec


for fmt, rec in sorted(representatives.items()):

    print("\n" + "=" * 100)
    print("FORMAT:", fmt)
    print("PROJECT:", rec["ProjectSlug"])
    print(
        "CONDITION:",
        rec["ConditionDir"].relative_to(
            RAW_ROOT / rec["ProjectSlug"]
        )
    )

    ranking_path = rec["RankingPath"]
    run_path = rec["RunPath"]

    # ------------------------------------------------------------
    # Read representative ranking file
    # ------------------------------------------------------------

    if ranking_path.suffix == ".parquet":
        rankings = pd.read_parquet(ranking_path)
    else:
        rankings = pd.read_csv(ranking_path)

    runs = pd.read_csv(run_path)

    print("\nRanking columns:")
    print(list(rankings.columns))

    print("\nProject-run columns:")
    print(list(runs.columns))

    # Technique column discovery
    ranking_tech_candidates = [
        c for c in rankings.columns
        if c.lower() in {
            "technique",
            "method",
            "model",
            "approach",
        }
    ]

    run_tech_candidates = [
        c for c in runs.columns
        if c.lower() in {
            "technique",
            "method",
            "model",
            "approach",
        }
    ]

    if not ranking_tech_candidates:
        print("\nWARNING: no obvious ranking technique column")
        display(rankings.head())
        continue

    if not run_tech_candidates:
        print("\nWARNING: no obvious project-run technique column")
        display(runs.head())
        continue

    rtc = ranking_tech_candidates[0]
    ptc = run_tech_candidates[0]

    latest_rankings = rankings[
        rankings[rtc].astype(str).str.lower()
        == "latestfail"
    ].copy()

    latest_runs = runs[
        runs[ptc].astype(str).str.lower()
        == "latestfail"
    ].copy()

    print(
        f"\nLatestFail ranking rows: "
        f"{len(latest_rankings)}"
    )

    print(
        f"LatestFail project-run rows: "
        f"{len(latest_runs)}"
    )

    print("\nLatestFail project-run record:")
    display(latest_runs)

    # Show useful ranking columns if they exist
    useful = [
        c for c in [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Score",
            "Rank",
            "CleanFailure",
            "Duration",
        ]
        if c in latest_rankings.columns
    ]

    print("\nFirst 10 LatestFail ranking rows:")
    display(
        latest_rankings[useful].head(10)
        if useful
        else latest_rankings.head(10)
    )

    if "Score" in latest_rankings.columns:
        scores = pd.to_numeric(
            latest_rankings["Score"],
            errors="coerce",
        )

        print(
            "\nScore == +1 rows:",
            int((scores == 1.0).sum())
        )

        print(
            "Score min/max:",
            float(scores.min()),
            float(scores.max()),
        )


print("\n=== END STEP 3C ===")

=== UNIVERSAL SOURCE ADAPTER DISCOVERY ===

Projects discovered: 24
Total conditions discovered: 6480

=== FORMAT COUNTS ===
  270 conditions | rankings.csv.gz + project_run_metrics.csv
 3780 conditions | rankings.csv.gz + project_runs.csv
  540 conditions | rankings.parquet + project_run.csv
 1890 conditions | rankings.parquet + project_run_metrics.csv

=== PROJECT COUNTS ===
Angel-ML__angel                               270
CompEvol__beast2                              270
EMResearch__EvoMaster                         270
JMRI__JMRI                                    270
SonarSource__sonarqube                        270
apache__airavata                              270
apache__curator                               270
apache__logging-log4j2                        270
apache__rocketmq                              270
apache__shardingsphere                        270
apache__sling                                 270
b2ihealthcare__snow-owl                       270
camunda__camunda-bpm

,Project,NoisePercent,RepetitionSeed,Technique,MeanAPFD,MeanAPFDc,SD_APFD,SD_APFDc,EvaluatedBuilds,TotalRankedTests,TotalFailures
0,optimatika@ojAlgo,0.0,1,LatestFail,0.934893,0.852401,0.158982,0.242511,16,2300,16



First 10 LatestFail ranking rows:


,Project,NoisePercent,RepetitionSeed,Technique,Build,Test,Score,Rank,Duration
11500,optimatika@ojAlgo,0,1,LatestFail,747791346,491,188.0,1,32.0
11501,optimatika@ojAlgo,0,1,LatestFail,747791346,996,186.0,2,24987.0
11502,optimatika@ojAlgo,0,1,LatestFail,747791346,1132,185.0,3,9.0
11503,optimatika@ojAlgo,0,1,LatestFail,747791346,626,172.0,4,2.0
11504,optimatika@ojAlgo,0,1,LatestFail,747791346,1003,168.0,5,144071.0
11505,optimatika@ojAlgo,0,1,LatestFail,747791346,477,166.0,6,228.0
11506,optimatika@ojAlgo,0,1,LatestFail,747791346,1173,163.0,7,2.0
11507,optimatika@ojAlgo,0,1,LatestFail,747791346,468,97.0,8,9.0
11508,optimatika@ojAlgo,0,1,LatestFail,747791346,478,1.0,9,125.0
11509,optimatika@ojAlgo,0,1,LatestFail,747791346,440,-1.0,10,2.0



Score == +1 rows: 16
Score min/max: -1.0 245.0

FORMAT: ('rankings.csv.gz', 'project_runs.csv')
PROJECT: EMResearch__EvoMaster
CONDITION: noise_00__seed_01

Ranking columns:
['ProjectNumber', 'Project', 'ProjectSlug', 'ConditionKey', 'NoisePercent', 'RepetitionSeed', 'Technique', 'Build', 'Test', 'Rank', 'Score', 'CleanVerdict', 'CleanFailure', 'Duration']

Project-run columns:
['ProjectNumber', 'Project', 'ProjectSlug', 'ConditionKey', 'NoisePercent', 'RepetitionSeed', 'Technique', 'EvaluationBuilds', 'ScoredFailingBuilds', 'EvaluationRows', 'EvaluationFailures', 'MeanAPFDc', 'MedianAPFDc', 'MeanAPFD', 'MedianAPFD']

LatestFail ranking rows: 4553
LatestFail project-run rows: 1

LatestFail project-run record:


...


Score == +1 rows: 0
Score min/max: -1.0 792.0

FORMAT: ('rankings.parquet', 'project_run_metrics.csv')
PROJECT: Angel-ML__angel
CONDITION: noise_000/seed_01

Ranking columns:
['Project', 'NoisePercent', 'RepetitionSeed', 'Build', 'Test', 'Verdict', 'Duration', 'build_order', 'Technique', 'Score', 'ActualFailure', 'Rank']

Project-run columns:
['Project', 'NoisePercent', 'RepetitionSeed', 'Technique', 'MeanAPFD', 'MeanAPFDc', 'SD_APFD', 'SD_APFDc', 'EvaluatedBuilds', 'TotalRankedTests', 'TotalFailures']

LatestFail ranking rows: 702
LatestFail project-run rows: 1

LatestFail project-run record:


,Project,NoisePercent,RepetitionSeed,Technique,MeanAPFD,MeanAPFDc,SD_APFD,SD_APFDc,EvaluatedBuilds,TotalRankedTests,TotalFailures
0,Angel-ML@angel,0.0,1,LatestFail,0.861206,0.872848,0.032388,0.028907,28,702,185



First 10 LatestFail ranking rows:


,Project,NoisePercent,RepetitionSeed,Technique,Build,Test,Score,Rank,Duration
3510,Angel-ML@angel,0.0,1,LatestFail,710784968,512,229.0,1,77967.0
3511,Angel-ML@angel,0.0,1,LatestFail,710784968,510,223.0,2,21612.0
3512,Angel-ML@angel,0.0,1,LatestFail,710784968,4821,166.0,3,30850.0
3513,Angel-ML@angel,0.0,1,LatestFail,710784968,506,-1.0,4,24093.0
3514,Angel-ML@angel,0.0,1,LatestFail,710784968,509,-1.0,5,32134.0
3515,Angel-ML@angel,0.0,1,LatestFail,710784968,511,-1.0,6,34850.0
3516,Angel-ML@angel,0.0,1,LatestFail,710784968,513,-1.0,7,20453.0
3517,Angel-ML@angel,0.0,1,LatestFail,710784968,541,-1.0,8,20327.0
3518,Angel-ML@angel,0.0,1,LatestFail,710784968,2161,-1.0,9,27.0
3519,Angel-ML@angel,0.0,1,LatestFail,710784968,2720,-1.0,10,1039.0



Score == +1 rows: 0
Score min/max: -1.0 307.0

=== END STEP 3C ===


In [10]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
RAW_ROOT = THESIS_ROOT / "Results" / "Raw"

RANKING_FILES = ["rankings.csv.gz", "rankings.parquet"]
RUN_FILES = [
    "project_runs.csv",
    "project_run.csv",
    "project_run_metrics.csv",
]

def detect_artifacts(directory):
    files = {p.name: p for p in directory.iterdir() if p.is_file()}

    ranking = next(
        (files[x] for x in RANKING_FILES if x in files),
        None
    )
    run = next(
        (files[x] for x in RUN_FILES if x in files),
        None
    )

    return (ranking, run) if ranking and run else None


def read_ranking(path):
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


rows = []

for project_root in sorted(
    p for p in RAW_ROOT.iterdir() if p.is_dir()
):

    candidates = []

    for current_root, dirs, files in os.walk(project_root):
        d = Path(current_root)
        detected = detect_artifacts(d)

        if detected is None:
            continue

        ranking_path, run_path = detected

        # Determine condition from project-run metadata,
        # avoiding assumptions about directory naming.
        run_df = pd.read_csv(run_path)

        if "NoisePercent" not in run_df.columns \
           or "RepetitionSeed" not in run_df.columns:
            continue

        noise_vals = pd.to_numeric(
            run_df["NoisePercent"],
            errors="coerce"
        ).dropna().unique()

        seed_vals = pd.to_numeric(
            run_df["RepetitionSeed"],
            errors="coerce"
        ).dropna().unique()

        if (
            len(noise_vals) == 1
            and len(seed_vals) == 1
            and int(noise_vals[0]) == 0
            and int(seed_vals[0]) == 1
        ):
            candidates.append(
                (d, ranking_path, run_path)
            )

    if len(candidates) != 1:
        raise RuntimeError(
            f"{project_root.name}: expected one 0%-seed1 condition, "
            f"found {len(candidates)}"
        )

    condition_dir, ranking_path, run_path = candidates[0]

    rankings = read_ranking(ranking_path)

    tech_col = next(
        c for c in rankings.columns
        if c.lower() in {"technique", "method", "model", "approach"}
    )

    lf = rankings[
        rankings[tech_col].astype(str).str.lower()
        == "latestfail"
    ].copy()

    scores = pd.to_numeric(
        lf["Score"],
        errors="coerce"
    )

    n = len(lf)
    plus1 = int(np.isclose(scores, 1.0).sum())
    minus1 = int(np.isclose(scores, -1.0).sum())

    # Rank positions of special score values.
    plus1_ranks = lf.loc[
        np.isclose(scores, 1.0),
        "Rank"
    ]

    minus1_ranks = lf.loc[
        np.isclose(scores, -1.0),
        "Rank"
    ]

    rows.append({
        "Project": project_root.name,
        "RankingFile": ranking_path.name,
        "RunFile": run_path.name,
        "Rows": n,
        "ScoreMin": float(scores.min()),
        "ScoreMax": float(scores.max()),
        "Plus1Rows": plus1,
        "Minus1Rows": minus1,

        "Plus1MeanRank":
            float(plus1_ranks.mean())
            if len(plus1_ranks) else np.nan,

        "Minus1MeanRank":
            float(minus1_ranks.mean())
            if len(minus1_ranks) else np.nan,

        "Plus1MeanRankPct":
            float(plus1_ranks.mean() / n * 100)
            if len(plus1_ranks) else np.nan,

        "Minus1MeanRankPct":
            float(minus1_ranks.mean() / n * 100)
            if len(minus1_ranks) else np.nan,
    })


audit = pd.DataFrame(rows)

# Classification based on observed score convention.
def classify(r):
    if (
        r["ScoreMax"] <= 1.0 + 1e-12
        and r["ScoreMin"] < -1.0
        and r["Plus1Rows"] > 0
    ):
        return "NEGATED_AGE__SENTINEL_PLUS1__SUSPECT"

    if (
        r["ScoreMin"] >= -1.0 - 1e-12
        and r["ScoreMax"] > 1.0
    ):
        return "POSITIVE_RECENCY__SENTINEL_MINUS1__LIKELY_OK"

    return "NEEDS_INSPECTION"


audit["Convention"] = audit.apply(
    classify,
    axis=1
)

audit = audit.sort_values(
    ["Convention", "Project"]
).reset_index(drop=True)

print("=== LATESTFAIL SCORE-CONVENTION AUDIT ===")
display(audit)

print("\n=== CONVENTION COUNTS ===")
print(audit["Convention"].value_counts())

print("\n=== SUSPECT PROJECTS ===")
display(
    audit[
        audit["Convention"]
        == "NEGATED_AGE__SENTINEL_PLUS1__SUSPECT"
    ][
        [
            "Project",
            "RankingFile",
            "Rows",
            "ScoreMin",
            "ScoreMax",
            "Plus1Rows",
            "Plus1MeanRankPct",
        ]
    ]
)

print("\n=== LEGACY / LIKELY-CORRECT PROJECTS ===")
display(
    audit[
        audit["Convention"]
        == "POSITIVE_RECENCY__SENTINEL_MINUS1__LIKELY_OK"
    ][
        [
            "Project",
            "RankingFile",
            "Rows",
            "ScoreMin",
            "ScoreMax",
            "Minus1Rows",
            "Minus1MeanRankPct",
        ]
    ]
)

print("\nNeeds inspection:")
print(
    audit.loc[
        audit["Convention"] == "NEEDS_INSPECTION",
        "Project"
    ].tolist()
)

KeyboardInterrupt: 

In [11]:
from pathlib import Path
import os
import re
import pandas as pd
import numpy as np

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
RAW_ROOT = THESIS_ROOT / "Results" / "Raw"

print("=== FAST LATESTFAIL SCORE-CONVENTION AUDIT ===\n")


# ---------------------------------------------------------------
# Recognise the historical 0%-noise / seed-1 directory formats
# ---------------------------------------------------------------

def is_zero_seed_one(rel_path):
    s = rel_path.as_posix()

    patterns = [
        # Modern:
        # noise_00__seed_01
        r"(?:^|/)noise_00__seed_0*1$",

        # Legacy grouped:
        # noise_000/seed_01
        # noise_000/seed_001
        r"(?:^|/)noise_000/seed_0*1$",

        # Beast2:
        # .../noise_000_seed_01
        r"(?:^|/)noise_000_seed_0*1$",
    ]

    return any(
        re.search(pattern, s)
        for pattern in patterns
    )


# ---------------------------------------------------------------
# Find exactly ONE 0%-seed1 ranking artifact per project
# WITHOUT opening CSV metadata files.
# ---------------------------------------------------------------

selected_files = []

project_roots = sorted(
    p for p in RAW_ROOT.iterdir()
    if p.is_dir()
)

for project_root in project_roots:

    matches = []

    for current_root, dirs, files in os.walk(project_root):

        current_dir = Path(current_root)
        rel = current_dir.relative_to(project_root)

        if not is_zero_seed_one(rel):
            continue

        if "rankings.csv.gz" in files:
            matches.append(
                current_dir / "rankings.csv.gz"
            )

        if "rankings.parquet" in files:
            matches.append(
                current_dir / "rankings.parquet"
            )

    if len(matches) != 1:
        raise RuntimeError(
            f"{project_root.name}: expected exactly one "
            f"0%-seed1 ranking file, found {len(matches)}.\n"
            f"Matches: {matches}"
        )

    selected_files.append(
        (project_root.name, matches[0])
    )


print(
    "Projects with 0%-seed1 ranking found:",
    len(selected_files)
)

if len(selected_files) != 24:
    raise RuntimeError(
        f"Expected 24 projects, found {len(selected_files)}"
    )


# ---------------------------------------------------------------
# Read ONLY Technique, Score and Rank
# ---------------------------------------------------------------

rows = []

for i, (project, ranking_path) in enumerate(
    selected_files,
    start=1,
):

    print(
        f"[{i:02d}/24] {project}",
        flush=True
    )

    if ranking_path.name.endswith(".parquet"):

        df = pd.read_parquet(
            ranking_path,
            columns=[
                "Technique",
                "Score",
                "Rank",
            ],
        )

    else:

        df = pd.read_csv(
            ranking_path,
            usecols=[
                "Technique",
                "Score",
                "Rank",
            ],
        )

    lf = df[
        df["Technique"]
        .astype(str)
        .str.lower()
        == "latestfail"
    ].copy()

    if lf.empty:
        raise RuntimeError(
            f"{project}: LatestFail rows missing"
        )

    scores = pd.to_numeric(
        lf["Score"],
        errors="coerce",
    )

    ranks = pd.to_numeric(
        lf["Rank"],
        errors="coerce",
    )

    plus1_mask = np.isclose(
        scores,
        1.0,
        rtol=0,
        atol=1e-12,
    )

    minus1_mask = np.isclose(
        scores,
        -1.0,
        rtol=0,
        atol=1e-12,
    )

    plus1_ranks = ranks[plus1_mask]
    minus1_ranks = ranks[minus1_mask]

    rows.append({
        "Project": project,
        "RankingFile": ranking_path.name,

        "Rows": len(lf),

        "ScoreMin":
            float(scores.min()),

        "ScoreMax":
            float(scores.max()),

        "Plus1Rows":
            int(plus1_mask.sum()),

        "Minus1Rows":
            int(minus1_mask.sum()),

        "Plus1MeanRankPct":
            (
                float(
                    plus1_ranks.mean()
                    / len(lf)
                    * 100
                )
                if len(plus1_ranks)
                else np.nan
            ),

        "Minus1MeanRankPct":
            (
                float(
                    minus1_ranks.mean()
                    / len(lf)
                    * 100
                )
                if len(minus1_ranks)
                else np.nan
            ),
    })

    # Free memory immediately.
    del df, lf


audit = pd.DataFrame(rows)


# ---------------------------------------------------------------
# Classify observed score convention
# ---------------------------------------------------------------

def classify(row):

    # Later pipeline:
    # ordinary scores <= 0,
    # sentinel became +1.
    if (
        row["ScoreMax"] <= 1.0 + 1e-12
        and row["ScoreMin"] < -1.0
        and row["Plus1Rows"] > 0
    ):
        return (
            "NEGATED_AGE__"
            "SENTINEL_PLUS1__SUSPECT"
        )

    # Earlier pipeline:
    # normal priority scores are positive,
    # sentinel remains -1 and therefore sorts last.
    if (
        row["ScoreMin"] >= -1.0 - 1e-12
        and row["ScoreMax"] > 1.0
    ):
        return (
            "POSITIVE_RECENCY__"
            "SENTINEL_MINUS1__LIKELY_OK"
        )

    return "NEEDS_INSPECTION"


audit["Convention"] = audit.apply(
    classify,
    axis=1,
)

audit = audit.sort_values(
    [
        "Convention",
        "Project",
    ]
).reset_index(drop=True)


print("\n" + "=" * 90)
print("CONVENTION COUNTS")
print("=" * 90)

print(
    audit["Convention"].value_counts()
)


print("\n=== SUSPECT PROJECTS ===")

suspect = audit[
    audit["Convention"]
    ==
    "NEGATED_AGE__SENTINEL_PLUS1__SUSPECT"
]

display(
    suspect[
        [
            "Project",
            "RankingFile",
            "Rows",
            "ScoreMin",
            "ScoreMax",
            "Plus1Rows",
            "Plus1MeanRankPct",
        ]
    ]
)


print("\n=== LEGACY / LIKELY-CORRECT PROJECTS ===")

legacy = audit[
    audit["Convention"]
    ==
    "POSITIVE_RECENCY__SENTINEL_MINUS1__LIKELY_OK"
]

display(
    legacy[
        [
            "Project",
            "RankingFile",
            "Rows",
            "ScoreMin",
            "ScoreMax",
            "Minus1Rows",
            "Minus1MeanRankPct",
        ]
    ]
)


needs = audit[
    audit["Convention"]
    == "NEEDS_INSPECTION"
]

print("\nNeeds inspection:")
print(
    needs["Project"].tolist()
)

print("\n=== FAST AUDIT COMPLETE ===")

=== FAST LATESTFAIL SCORE-CONVENTION AUDIT ===

Projects with 0%-seed1 ranking found: 24
[01/24] Angel-ML__angel
[02/24] CompEvol__beast2
[03/24] EMResearch__EvoMaster
[04/24] JMRI__JMRI
[05/24] SonarSource__sonarqube
[06/24] apache__airavata
[07/24] apache__curator
[08/24] apache__logging-log4j2
[09/24] apache__rocketmq
[10/24] apache__shardingsphere
[11/24] apache__sling
[12/24] b2ihealthcare__snow-owl
[13/24] camunda__camunda-bpm-platform
[14/24] cantaloupe-project__cantaloupe
[15/24] eclipse__jetty.project
[16/24] eclipse__paho.mqtt.java
[17/24] eclipse__steady
[18/24] facebook__buck
[19/24] jcabi__jcabi-github
[20/24] optimatika__ojAlgo
[21/24] spring-cloud__spring-cloud-dataflow
[22/24] thinkaurelius__titan
[23/24] yamcs__Yamcs
[24/24] zolyfarkas__spf4j

CONVENTION COUNTS
Convention
NEGATED_AGE__SENTINEL_PLUS1__SUSPECT            14
POSITIVE_RECENCY__SENTINEL_MINUS1__LIKELY_OK    10
Name: count, dtype: int64

=== SUSPECT PROJECTS ===


,Project,RankingFile,Rows,ScoreMin,ScoreMax,Plus1Rows,Plus1MeanRankPct
0,EMResearch__EvoMaster,rankings.csv.gz,4553,-511.0,1.0,2214,0.605614
1,JMRI__JMRI,rankings.csv.gz,107144,-1302.0,1.0,104491,2.032867
2,SonarSource__sonarqube,rankings.csv.gz,18854,-3836.0,1.0,17795,3.357489
3,apache__curator,rankings.csv.gz,106,-329.0,1.0,83,32.052739
4,apache__logging-log4j2,rankings.csv.gz,22156,-382.0,1.0,21549,1.266175
5,apache__rocketmq,rankings.csv.gz,1641,-434.0,1.0,1433,5.490712
6,apache__shardingsphere,rankings.csv.gz,13007,-866.0,1.0,12320,2.307665
7,apache__sling,rankings.csv.gz,6018,-1204.0,1.0,5452,1.153293
8,cantaloupe-project__cantaloupe,rankings.csv.gz,2348,-412.0,1.0,1582,2.830528
9,eclipse__steady,rankings.csv.gz,1056,-348.0,1.0,997,3.981262



=== LEGACY / LIKELY-CORRECT PROJECTS ===


,Project,RankingFile,Rows,ScoreMin,ScoreMax,Minus1Rows,Minus1MeanRankPct
14,Angel-ML__angel,rankings.parquet,702,-1.0,307.0,512,2.360722
15,CompEvol__beast2,rankings.parquet,3678,-1.0,414.0,2539,1.272985
16,apache__airavata,rankings.parquet,249,-1.0,207.0,84,16.838784
17,b2ihealthcare__snow-owl,rankings.parquet,581,-1.0,247.0,156,6.146564
18,camunda__camunda-bpm-platform,rankings.parquet,18503,-1.0,792.0,14903,2.100335
19,eclipse__jetty.project,rankings.parquet,4079,-1.0,191.0,3632,1.477793
20,eclipse__paho.mqtt.java,rankings.parquet,980,-1.0,358.0,780,3.061224
21,optimatika__ojAlgo,rankings.csv.gz,2300,-1.0,245.0,2145,3.357292
22,spring-cloud__spring-cloud-dataflow,rankings.parquet,2611,-1.0,391.0,1802,2.504005
23,thinkaurelius__titan,rankings.parquet,363,-1.0,383.0,266,7.174962



Needs inspection:
[]

=== FAST AUDIT COMPLETE ===


In [12]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import hashlib
import json
import time

# ============================================================================
# LATESTFAIL SENTINEL CORRECTION — PROJECT 16 FULL 270-CONDITION PILOT
#
# Project: apache@rocketmq
#
# READ ONLY:
#   Results/Raw/apache__rocketmq
#   Results/Aggregated/apache__rocketmq
#
# WRITES ONLY:
#   Results/Corrections/LatestFail_Sentinel_Fix_v2/
#
# NO ML models are retrained.
# NO frozen outputs are modified.
# ============================================================================

ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
RESULTS = ROOT / "Results"

PROJECT_NUMBER = 16
PROJECT_NAME = "apache@rocketmq"
PROJECT_SLUG = "apache__rocketmq"

RAW_ROOT = RESULTS / "Raw" / PROJECT_SLUG

FROZEN_RUNS_PATH = (
    RESULTS
    / "Aggregated"
    / PROJECT_SLUG
    / "ROCKETMQ_step5b"
    / "ROCKETMQ_revalidated_project_runs.csv"
)

CORRECTION_ROOT = (
    RESULTS
    / "Corrections"
    / "LatestFail_Sentinel_Fix_v2"
)

OUT = CORRECTION_ROOT / PROJECT_SLUG
OUT.mkdir(parents=True, exist_ok=True)

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
SEEDS = list(range(1, 31))

EXPECTED_CONDITIONS = 270
TOL = 1e-10


# ----------------------------------------------------------------------------
# Metrics — already validated against the frozen RocketMQ seed-1 results
# ----------------------------------------------------------------------------

def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)

    n = len(failures)
    m = int(failures.sum())

    if n == 0 or m == 0:
        return np.nan

    positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - positions.sum() / (n * m)
        + 1.0 / (2.0 * n)
    )


def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    n = len(failures)
    m = int(failures.sum())

    if n == 0 or m == 0:
        return np.nan

    total_duration = durations.sum()

    if not np.isfinite(total_duration) or total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate(
        [
            np.array([0.0]),
            np.cumsum(durations)[:-1]
        ]
    )

    failure_mask = failures == 1

    midpoint_times = (
        cumulative_before[failure_mask]
        + 0.5 * durations[failure_mask]
    )

    return float(
        1.0
        - np.mean(midpoint_times / total_duration)
    )


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)

            if not b:
                break

            h.update(b)

    return h.hexdigest()


def compute_build_metrics(df, rank_col):
    rows = []

    for build, g in df.groupby("Build", sort=False):

        g = g.sort_values(
            rank_col,
            kind="mergesort"
        )

        failures = int(
            g["CleanFailure"].sum()
        )

        # The frozen experiment scores only failing evaluation builds.
        if failures == 0:
            continue

        rows.append(
            {
                "Build": build,
                "Tests": int(len(g)),
                "Failures": failures,
                "TotalDuration": float(
                    g["Duration"].sum()
                ),
                "APFD": calculate_apfd(
                    g["CleanFailure"]
                ),
                "APFDc": calculate_apfdc(
                    g["CleanFailure"],
                    g["Duration"]
                ),
            }
        )

    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# Validate required inputs
# ----------------------------------------------------------------------------

assert RAW_ROOT.is_dir(), RAW_ROOT
assert FROZEN_RUNS_PATH.is_file(), FROZEN_RUNS_PATH

frozen_all = pd.read_csv(
    FROZEN_RUNS_PATH,
    low_memory=False
)

frozen_lf = frozen_all[
    frozen_all["Technique"].astype(str)
    == "LatestFail"
].copy()

assert len(frozen_lf) == 270, (
    f"Expected 270 frozen LatestFail rows, got {len(frozen_lf)}"
)

frozen_lf["NoisePercent"] = pd.to_numeric(
    frozen_lf["NoisePercent"]
).astype(int)

frozen_lf["RepetitionSeed"] = pd.to_numeric(
    frozen_lf["RepetitionSeed"]
).astype(int)

frozen_lookup = frozen_lf.set_index(
    [
        "NoisePercent",
        "RepetitionSeed"
    ]
)

print("=" * 90)
print("PROJECT 16 LATESTFAIL FULL CORRECTION PILOT")
print("=" * 90)
print("Frozen LatestFail rows:", len(frozen_lf))
print("Correction output:", OUT)
print()


# ----------------------------------------------------------------------------
# Process 270 conditions
# ----------------------------------------------------------------------------

validation_rows = []
corrected_run_rows = []
corrected_build_frames = []

start = time.perf_counter()
condition_counter = 0


for noise in NOISE_LEVELS:

    for seed in SEEDS:

        condition_counter += 1

        condition_key = (
            f"noise_{noise:02d}__seed_{seed:02d}"
        )

        ranking_path = (
            RAW_ROOT
            / condition_key
            / "rankings.csv.gz"
        )

        if not ranking_path.is_file():
            raise FileNotFoundError(
                f"Missing ranking file: {ranking_path}"
            )

        # Read only the columns required for LatestFail correction.
        rankings = pd.read_csv(
            ranking_path,
            usecols=[
                "Technique",
                "Build",
                "Test",
                "Rank",
                "Score",
                "CleanFailure",
                "Duration",
            ],
            low_memory=False,
        )

        lf = rankings[
            rankings["Technique"].astype(str)
            == "LatestFail"
        ].copy()

        del rankings

        if lf.empty:
            raise RuntimeError(
                f"{condition_key}: no LatestFail rows"
            )

        lf["Score"] = pd.to_numeric(
            lf["Score"],
            errors="raise"
        )

        lf["Rank"] = pd.to_numeric(
            lf["Rank"],
            errors="raise"
        ).astype(int)

        lf["CleanFailure"] = pd.to_numeric(
            lf["CleanFailure"],
            errors="raise"
        ).astype(int)

        lf["Duration"] = pd.to_numeric(
            lf["Duration"],
            errors="raise"
        ).astype(float)

        # --------------------------------------------------------------------
        # Validate this condition belongs to the affected score convention.
        #
        # Normal prior-failure scores: <= 0
        # Never-failed sentinel:
        #   REC_LastFailureAge = -1
        #   old Score = -(-1) = +1
        # --------------------------------------------------------------------

        score_max = float(lf["Score"].max())

        if score_max > 1.0 + 1e-12:
            raise RuntimeError(
                f"{condition_key}: unexpected LatestFail "
                f"score convention; max={score_max}"
            )

        sentinel = np.isclose(
            lf["Score"].to_numpy(dtype=float),
            1.0,
            rtol=0,
            atol=1e-12,
        )

        lf["NeverFailed"] = sentinel

        sentinel_rows = int(
            lf["NeverFailed"].sum()
        )

        # --------------------------------------------------------------------
        # Reconstruct OLD metrics exactly as frozen
        # --------------------------------------------------------------------

        old_build = compute_build_metrics(
            lf,
            "Rank"
        )

        old_mean_apfd = float(
            old_build["APFD"].mean()
        )

        old_mean_apfdc = float(
            old_build["APFDc"].mean()
        )

        old_median_apfd = float(
            old_build["APFD"].median()
        )

        old_median_apfdc = float(
            old_build["APFDc"].median()
        )

        frozen = frozen_lookup.loc[
            (noise, seed)
        ]

        frozen_apfd = float(
            frozen["MeanAPFD"]
        )

        frozen_apfdc = float(
            frozen["MeanAPFDc"]
        )

        old_apfd_match = np.isclose(
            old_mean_apfd,
            frozen_apfd,
            rtol=TOL,
            atol=1e-12,
        )

        old_apfdc_match = np.isclose(
            old_mean_apfdc,
            frozen_apfdc,
            rtol=TOL,
            atol=1e-12,
        )

        if not old_apfd_match:
            raise RuntimeError(
                f"{condition_key}: OLD APFD reconstruction mismatch\n"
                f"reconstructed={old_mean_apfd}\n"
                f"frozen={frozen_apfd}"
            )

        if not old_apfdc_match:
            raise RuntimeError(
                f"{condition_key}: OLD APFDc reconstruction mismatch\n"
                f"reconstructed={old_mean_apfdc}\n"
                f"frozen={frozen_apfdc}"
            )

        # --------------------------------------------------------------------
        # Correct LatestFail:
        #
        # 1. tests with a real previous failure first;
        # 2. among those, larger old score first
        #    (= more recent failure because Score=-Age);
        # 3. never-failed sentinel rows last;
        # 4. Test is deterministic tie-break.
        # --------------------------------------------------------------------

        corrected = lf.sort_values(
            [
                "Build",
                "NeverFailed",
                "Score",
                "Test",
            ],
            ascending=[
                True,
                True,    # False before True
                False,   # higher real score first
                True,    # deterministic tie-break
            ],
            kind="mergesort",
        ).copy()

        corrected["CorrectedRank"] = (
            corrected
            .groupby(
                "Build",
                sort=False
            )
            .cumcount()
            .add(1)
        )

        # --------------------------------------------------------------------
        # Strong within-build validation:
        # once a sentinel appears, no non-sentinel may follow it.
        # --------------------------------------------------------------------

        for build, g in corrected.groupby(
            "Build",
            sort=False
        ):
            flags = g[
                "NeverFailed"
            ].to_numpy(dtype=bool)

            if np.any(
                flags[:-1]
                & (~flags[1:])
            ):
                raise RuntimeError(
                    f"{condition_key}/{build}: "
                    "sentinel ordering validation failed"
                )

        # --------------------------------------------------------------------
        # Corrected metrics
        # --------------------------------------------------------------------

        corrected_build = compute_build_metrics(
            corrected,
            "CorrectedRank"
        )

        corrected_mean_apfd = float(
            corrected_build["APFD"].mean()
        )

        corrected_mean_apfdc = float(
            corrected_build["APFDc"].mean()
        )

        corrected_median_apfd = float(
            corrected_build["APFD"].median()
        )

        corrected_median_apfdc = float(
            corrected_build["APFDc"].median()
        )

        # --------------------------------------------------------------------
        # Preserve build metrics as compact correction evidence
        # --------------------------------------------------------------------

        corrected_build.insert(
            0,
            "RepetitionSeed",
            seed
        )

        corrected_build.insert(
            0,
            "NoisePercent",
            noise
        )

        corrected_build.insert(
            0,
            "ConditionKey",
            condition_key
        )

        corrected_build.insert(
            0,
            "ProjectSlug",
            PROJECT_SLUG
        )

        corrected_build.insert(
            0,
            "Project",
            PROJECT_NAME
        )

        corrected_build.insert(
            0,
            "ProjectNumber",
            PROJECT_NUMBER
        )

        corrected_build.insert(
            6,
            "Technique",
            "LatestFail"
        )

        corrected_build_frames.append(
            corrected_build
        )

        # --------------------------------------------------------------------
        # Corrected project-run record
        # --------------------------------------------------------------------

        corrected_run_rows.append(
            {
                "ProjectNumber":
                    PROJECT_NUMBER,

                "Project":
                    PROJECT_NAME,

                "ProjectSlug":
                    PROJECT_SLUG,

                "ConditionKey":
                    condition_key,

                "NoisePercent":
                    noise,

                "RepetitionSeed":
                    seed,

                "Technique":
                    "LatestFail",

                "EvaluationBuilds":
                    int(
                        lf["Build"].nunique()
                    ),

                "ScoredFailingBuilds":
                    int(
                        len(corrected_build)
                    ),

                "EvaluationRows":
                    int(
                        len(lf)
                    ),

                "EvaluationFailures":
                    int(
                        lf["CleanFailure"].sum()
                    ),

                "MeanAPFDc":
                    corrected_mean_apfdc,

                "MedianAPFDc":
                    corrected_median_apfdc,

                "MeanAPFD":
                    corrected_mean_apfd,

                "MedianAPFD":
                    corrected_median_apfd,
            }
        )

        validation_rows.append(
            {
                "ProjectNumber":
                    PROJECT_NUMBER,

                "Project":
                    PROJECT_NAME,

                "ProjectSlug":
                    PROJECT_SLUG,

                "ConditionKey":
                    condition_key,

                "NoisePercent":
                    noise,

                "RepetitionSeed":
                    seed,

                "LatestFailRows":
                    int(len(lf)),

                "SentinelRows":
                    sentinel_rows,

                "SentinelPercent":
                    100.0
                    * sentinel_rows
                    / len(lf),

                "FrozenMeanAPFD":
                    frozen_apfd,

                "ReconstructedOldMeanAPFD":
                    old_mean_apfd,

                "OldAPFDMatch":
                    bool(old_apfd_match),

                "CorrectedMeanAPFD":
                    corrected_mean_apfd,

                "APFDChange":
                    corrected_mean_apfd
                    - frozen_apfd,

                "FrozenMeanAPFDc":
                    frozen_apfdc,

                "ReconstructedOldMeanAPFDc":
                    old_mean_apfdc,

                "OldAPFDcMatch":
                    bool(old_apfdc_match),

                "CorrectedMeanAPFDc":
                    corrected_mean_apfdc,

                "APFDcChange":
                    corrected_mean_apfdc
                    - frozen_apfdc,
            }
        )

        # --------------------------------------------------------------------
        # Anchor against the diagnostic we already independently ran
        # --------------------------------------------------------------------

        if noise == 0 and seed == 1:

            if not np.isclose(
                corrected_mean_apfd,
                0.976398,
                rtol=0,
                atol=1e-6,
            ):
                raise RuntimeError(
                    "0%-seed1 corrected APFD does not match "
                    "the previous diagnostic anchor."
                )

            if not np.isclose(
                corrected_mean_apfdc,
                0.894954,
                rtol=0,
                atol=1e-6,
            ):
                raise RuntimeError(
                    "0%-seed1 corrected APFDc does not match "
                    "the previous diagnostic anchor."
                )

        # Progress every 30 conditions.
        if (
            condition_counter % 30 == 0
            or condition_counter
            == EXPECTED_CONDITIONS
        ):
            elapsed = (
                time.perf_counter()
                - start
            )

            print(
                f"Progress: "
                f"{condition_counter:3d}/270 | "
                f"noise={noise:02d}% | "
                f"elapsed={elapsed:.1f}s",
                flush=True,
            )


# ----------------------------------------------------------------------------
# Final compact outputs
# ----------------------------------------------------------------------------

validation = pd.DataFrame(
    validation_rows
).sort_values(
    [
        "NoisePercent",
        "RepetitionSeed",
    ]
).reset_index(drop=True)

corrected_runs = pd.DataFrame(
    corrected_run_rows
).sort_values(
    [
        "NoisePercent",
        "RepetitionSeed",
    ]
).reset_index(drop=True)

corrected_builds = pd.concat(
    corrected_build_frames,
    ignore_index=True
)

assert len(validation) == 270
assert len(corrected_runs) == 270

assert validation[
    "OldAPFDMatch"
].all()

assert validation[
    "OldAPFDcMatch"
].all()


# ----------------------------------------------------------------------------
# Noise-level summary for this project
# ----------------------------------------------------------------------------

noise_summary = (
    corrected_runs
    .groupby(
        "NoisePercent",
        as_index=False
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),
        CorrectedMeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),
        CorrectedSDAPFDc=(
            "MeanAPFDc",
            "std",
        ),
        CorrectedMeanAPFD=(
            "MeanAPFD",
            "mean",
        ),
        CorrectedSDAPFD=(
            "MeanAPFD",
            "std",
        ),
    )
)


old_noise = (
    frozen_lf
    .groupby(
        "NoisePercent",
        as_index=False
    )
    .agg(
        FrozenMeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),
        FrozenMeanAPFD=(
            "MeanAPFD",
            "mean",
        ),
    )
)

noise_summary = old_noise.merge(
    noise_summary,
    on="NoisePercent",
    how="one_to_one",
)

noise_summary["APFDcCorrection"] = (
    noise_summary[
        "CorrectedMeanAPFDc"
    ]
    - noise_summary[
        "FrozenMeanAPFDc"
    ]
)

noise_summary["APFDCorrection"] = (
    noise_summary[
        "CorrectedMeanAPFD"
    ]
    - noise_summary[
        "FrozenMeanAPFD"
    ]
)


# ----------------------------------------------------------------------------
# Write ONLY correction artifacts
# ----------------------------------------------------------------------------

VALIDATION_PATH = (
    OUT
    / "ROCKETMQ_latestfail_correction_validation.csv"
)

RUNS_PATH = (
    OUT
    / "ROCKETMQ_latestfail_corrected_project_runs.csv"
)

BUILDS_PATH = (
    OUT
    / "ROCKETMQ_latestfail_corrected_build_metrics.csv"
)

SUMMARY_PATH = (
    OUT
    / "ROCKETMQ_latestfail_corrected_noise_summary.csv"
)

MANIFEST_PATH = (
    OUT
    / "ROCKETMQ_latestfail_correction_manifest.json"
)

validation.to_csv(
    VALIDATION_PATH,
    index=False
)

corrected_runs.to_csv(
    RUNS_PATH,
    index=False
)

corrected_builds.to_csv(
    BUILDS_PATH,
    index=False
)

noise_summary.to_csv(
    SUMMARY_PATH,
    index=False
)

manifest = {
    "CorrectionID":
        "LatestFail_Sentinel_Fix_v2",

    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CreatedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "SourceRawRoot":
        str(RAW_ROOT),

    "SourceFrozenProjectRuns":
        str(FROZEN_RUNS_PATH),

    "FrozenSourceModification":
        False,

    "ConditionsProcessed":
        int(len(corrected_runs)),

    "OldAPFDValidationPass":
        bool(
            validation[
                "OldAPFDMatch"
            ].all()
        ),

    "OldAPFDcValidationPass":
        bool(
            validation[
                "OldAPFDcMatch"
            ].all()
        ),

    "Correction":
        (
            "For the affected later LatestFail implementation, "
            "REC_LastFailureAge=-1 indicated no prior failure. "
            "Negating this value produced Score=+1, which placed "
            "never-failed tests before tests with genuine prior "
            "failures. Sentinel rows are now placed after all "
            "tests with prior failures while preserving the "
            "original descending score order and Test tie-break "
            "for all non-sentinel rows."
        ),

    "ValidationSHA256":
        sha256_file(
            VALIDATION_PATH
        ),

    "CorrectedRunsSHA256":
        sha256_file(
            RUNS_PATH
        ),

    "CorrectedBuildsSHA256":
        sha256_file(
            BUILDS_PATH
        ),

    "NoiseSummarySHA256":
        sha256_file(
            SUMMARY_PATH
        ),
}

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        sort_keys=True,
    )


elapsed = time.perf_counter() - start

print("\n" + "=" * 90)
print("PROJECT 16 LATESTFAIL CORRECTION PILOT COMPLETE")
print("=" * 90)

print(
    "Conditions processed:",
    len(corrected_runs),
)

print(
    "All frozen APFD values reconstructed:",
    validation["OldAPFDMatch"].all(),
)

print(
    "All frozen APFDc values reconstructed:",
    validation["OldAPFDcMatch"].all(),
)

print(
    "Total elapsed seconds:",
    round(elapsed, 2),
)

print("\n=== ROCKETMQ OLD VS CORRECTED LATESTFAIL ===")
display(noise_summary)

print("\nOutput directory:")
print(OUT)

PROJECT 16 LATESTFAIL FULL CORRECTION PILOT
Frozen LatestFail rows: 270
Correction output: /content/drive/MyDrive/Thesis_Experiment/Results/Corrections/LatestFail_Sentinel_Fix_v2/apache__rocketmq

Progress:  30/270 | noise=00% | elapsed=11.5s
Progress:  60/270 | noise=05% | elapsed=22.7s
Progress:  90/270 | noise=10% | elapsed=34.0s
Progress: 120/270 | noise=15% | elapsed=45.3s
Progress: 150/270 | noise=20% | elapsed=56.3s
Progress: 180/270 | noise=25% | elapsed=66.9s
Progress: 210/270 | noise=30% | elapsed=78.0s
Progress: 240/270 | noise=40% | elapsed=88.5s
Progress: 270/270 | noise=50% | elapsed=99.1s


ValueError: do not recognize join method one_to_one

In [13]:
# ============================================================================
# PROJECT 16 LATESTFAIL CORRECTION — RECOVERY AFTER MERGE ERROR
#
# Do NOT rerun the 270-condition processing.
# This continues from the variables already created in memory:
#   validation
#   corrected_runs
#   corrected_builds
#   frozen_lf
#   OUT
#   sha256_file
# ============================================================================

from datetime import datetime, timezone
import pandas as pd
import numpy as np
import json
import time

print("=" * 90)
print("RESUMING PROJECT 16 LATESTFAIL CORRECTION")
print("=" * 90)

# ---------------------------------------------------------------------------
# Confirm that the expensive part really completed and is still in memory
# ---------------------------------------------------------------------------

required_objects = [
    "validation",
    "corrected_runs",
    "corrected_builds",
    "frozen_lf",
    "OUT",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The Colab runtime no longer contains the completed correction "
        f"objects: {missing_objects}\n"
        "If the runtime was restarted, the 270-condition pilot must be "
        "rerun with the corrected merge line."
    )

assert len(validation) == 270, (
    f"Expected 270 validation rows, found {len(validation)}"
)

assert len(corrected_runs) == 270, (
    f"Expected 270 corrected run rows, found {len(corrected_runs)}"
)

assert validation["OldAPFDMatch"].all(), (
    "At least one frozen APFD reconstruction failed."
)

assert validation["OldAPFDcMatch"].all(), (
    "At least one frozen APFDc reconstruction failed."
)

print("Completed conditions recovered:", len(corrected_runs))
print(
    "All frozen APFD reconstructions:",
    validation["OldAPFDMatch"].all()
)
print(
    "All frozen APFDc reconstructions:",
    validation["OldAPFDcMatch"].all()
)


# ---------------------------------------------------------------------------
# Corrected noise-level summary
# ---------------------------------------------------------------------------

corrected_noise = (
    corrected_runs
    .groupby(
        "NoisePercent",
        as_index=False
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),
        CorrectedMeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),
        CorrectedSDAPFDc=(
            "MeanAPFDc",
            "std",
        ),
        CorrectedMeanAPFD=(
            "MeanAPFD",
            "mean",
        ),
        CorrectedSDAPFD=(
            "MeanAPFD",
            "std",
        ),
    )
    .sort_values("NoisePercent")
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# Frozen/original noise-level summary
# ---------------------------------------------------------------------------

old_noise = (
    frozen_lf
    .groupby(
        "NoisePercent",
        as_index=False
    )
    .agg(
        FrozenMeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),
        FrozenMeanAPFD=(
            "MeanAPFD",
            "mean",
        ),
    )
    .sort_values("NoisePercent")
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# FIXED MERGE
#
# "one_to_one" belongs in validate=, NOT how=
# ---------------------------------------------------------------------------

noise_summary = old_noise.merge(
    corrected_noise,
    on="NoisePercent",
    how="inner",
    validate="one_to_one",
)

assert len(noise_summary) == 9, (
    f"Expected 9 noise levels, found {len(noise_summary)}"
)

assert set(noise_summary["NoisePercent"]) == {
    0, 5, 10, 15, 20, 25, 30, 40, 50
}


# ---------------------------------------------------------------------------
# Differences produced by the correction
# ---------------------------------------------------------------------------

noise_summary["APFDcCorrection"] = (
    noise_summary["CorrectedMeanAPFDc"]
    - noise_summary["FrozenMeanAPFDc"]
)

noise_summary["APFDCorrection"] = (
    noise_summary["CorrectedMeanAPFD"]
    - noise_summary["FrozenMeanAPFD"]
)


# ---------------------------------------------------------------------------
# Recheck our independently established seed-1 diagnostic anchors
# ---------------------------------------------------------------------------

anchor = validation[
    (validation["NoisePercent"] == 0)
    & (validation["RepetitionSeed"] == 1)
]

assert len(anchor) == 1

anchor = anchor.iloc[0]

assert np.isclose(
    float(anchor["CorrectedMeanAPFD"]),
    0.976398,
    rtol=0,
    atol=1e-6,
), (
    "0%-seed1 corrected APFD does not match prior diagnostic."
)

assert np.isclose(
    float(anchor["CorrectedMeanAPFDc"]),
    0.894954,
    rtol=0,
    atol=1e-6,
), (
    "0%-seed1 corrected APFDc does not match prior diagnostic."
)

print("\nIndependent RocketMQ seed-1 anchor: PASS")


# ---------------------------------------------------------------------------
# Save correction artifacts
# ---------------------------------------------------------------------------

VALIDATION_PATH = (
    OUT
    / "ROCKETMQ_latestfail_correction_validation.csv"
)

RUNS_PATH = (
    OUT
    / "ROCKETMQ_latestfail_corrected_project_runs.csv"
)

BUILDS_PATH = (
    OUT
    / "ROCKETMQ_latestfail_corrected_build_metrics.csv"
)

SUMMARY_PATH = (
    OUT
    / "ROCKETMQ_latestfail_corrected_noise_summary.csv"
)

MANIFEST_PATH = (
    OUT
    / "ROCKETMQ_latestfail_correction_manifest.json"
)


validation.to_csv(
    VALIDATION_PATH,
    index=False,
)

corrected_runs.to_csv(
    RUNS_PATH,
    index=False,
)

corrected_builds.to_csv(
    BUILDS_PATH,
    index=False,
)

noise_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)


manifest = {
    "CorrectionID":
        "LatestFail_Sentinel_Fix_v2",

    "ProjectNumber":
        16,

    "Project":
        "apache@rocketmq",

    "ProjectSlug":
        "apache__rocketmq",

    "CreatedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "FrozenSourceModification":
        False,

    "ConditionsProcessed":
        int(len(corrected_runs)),

    "NoiseLevels":
        [
            0, 5, 10, 15, 20,
            25, 30, 40, 50
        ],

    "SeedsPerNoiseLevel":
        30,

    "OldAPFDValidationPass":
        bool(
            validation["OldAPFDMatch"].all()
        ),

    "OldAPFDcValidationPass":
        bool(
            validation["OldAPFDcMatch"].all()
        ),

    "Seed1DiagnosticAnchorPass":
        True,

    "Correction":
        (
            "For the affected LatestFail implementation, "
            "REC_LastFailureAge=-1 represented no prior failure. "
            "The transformation Score=-REC_LastFailureAge converted "
            "this sentinel to +1 and therefore ranked never-failed "
            "tests ahead of tests with genuine prior failures. "
            "The correction places these sentinel rows after all "
            "tests with a recorded prior failure, while preserving "
            "the original descending score ordering and deterministic "
            "Test tie-breaking among non-sentinel rows."
        ),

    "ValidationSHA256":
        sha256_file(
            VALIDATION_PATH
        ),

    "CorrectedRunsSHA256":
        sha256_file(
            RUNS_PATH
        ),

    "CorrectedBuildsSHA256":
        sha256_file(
            BUILDS_PATH
        ),

    "NoiseSummarySHA256":
        sha256_file(
            SUMMARY_PATH
        ),
}


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        sort_keys=True,
    )


# ---------------------------------------------------------------------------
# Final report
# ---------------------------------------------------------------------------

print("\n" + "=" * 90)
print("PROJECT 16 LATESTFAIL CORRECTION PILOT — PASS")
print("=" * 90)

print(
    "Conditions processed:",
    len(corrected_runs)
)

print(
    "All frozen APFD values reconstructed:",
    validation["OldAPFDMatch"].all()
)

print(
    "All frozen APFDc values reconstructed:",
    validation["OldAPFDcMatch"].all()
)

print(
    "Noise levels summarized:",
    len(noise_summary)
)

print("\n=== ROCKETMQ OLD VS CORRECTED LATESTFAIL ===")
display(noise_summary)

print("\nSaved correction artifacts:")
print(" ", VALIDATION_PATH)
print(" ", RUNS_PATH)
print(" ", BUILDS_PATH)
print(" ", SUMMARY_PATH)
print(" ", MANIFEST_PATH)

print("\nCorrection directory:")
print(OUT)

RESUMING PROJECT 16 LATESTFAIL CORRECTION
Completed conditions recovered: 270
All frozen APFD reconstructions: True
All frozen APFDc reconstructions: True

Independent RocketMQ seed-1 anchor: PASS

PROJECT 16 LATESTFAIL CORRECTION PILOT — PASS
Conditions processed: 270
All frozen APFD values reconstructed: True
All frozen APFDc values reconstructed: True
Noise levels summarized: 9

=== ROCKETMQ OLD VS CORRECTED LATESTFAIL ===


,NoisePercent,FrozenMeanAPFDc,FrozenMeanAPFD,Seeds,CorrectedMeanAPFDc,CorrectedSDAPFDc,CorrectedMeanAPFD,CorrectedSDAPFD,APFDcCorrection,APFDCorrection
0,0,0.299580,0.103191,30,0.894954,0.000000,0.976398,0.000000,0.595373,0.873206
1,5,0.816818,0.847187,30,0.822374,0.061016,0.855470,0.058358,0.005556,0.008282
2,10,0.829873,0.851964,30,0.834259,0.068621,0.857605,0.066187,0.004386,0.005642
3,15,0.847495,0.865229,30,0.851319,0.069668,0.869937,0.069349,0.003824,0.004708
4,20,0.859006,0.877259,30,0.862654,0.062125,0.881363,0.063009,0.003648,0.004104
5,25,0.868010,0.882133,30,0.871422,0.051842,0.885777,0.056987,0.003413,0.003643
6,30,0.876794,0.891827,30,0.880110,0.043862,0.895393,0.047261,0.003316,0.003566
7,40,0.876081,0.893668,30,0.879300,0.050916,0.897156,0.052587,0.003219,0.003488
8,50,0.887502,0.907004,30,0.890720,0.045806,0.910492,0.041391,0.003219,0.003488



Saved correction artifacts:
  /content/drive/MyDrive/Thesis_Experiment/Results/Corrections/LatestFail_Sentinel_Fix_v2/apache__rocketmq/ROCKETMQ_latestfail_correction_validation.csv
  /content/drive/MyDrive/Thesis_Experiment/Results/Corrections/LatestFail_Sentinel_Fix_v2/apache__rocketmq/ROCKETMQ_latestfail_corrected_project_runs.csv
  /content/drive/MyDrive/Thesis_Experiment/Results/Corrections/LatestFail_Sentinel_Fix_v2/apache__rocketmq/ROCKETMQ_latestfail_corrected_build_metrics.csv
  /content/drive/MyDrive/Thesis_Experiment/Results/Corrections/LatestFail_Sentinel_Fix_v2/apache__rocketmq/ROCKETMQ_latestfail_corrected_noise_summary.csv
  /content/drive/MyDrive/Thesis_Experiment/Results/Corrections/LatestFail_Sentinel_Fix_v2/apache__rocketmq/ROCKETMQ_latestfail_correction_manifest.json

Correction directory:
/content/drive/MyDrive/Thesis_Experiment/Results/Corrections/LatestFail_Sentinel_Fix_v2/apache__rocketmq


In [15]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import json
import hashlib
import time

# =============================================================================
# LATESTFAIL SENTINEL FIX V2
# FULL CORRECTION FOR THE 14 AFFECTED MODERN-FORMAT PROJECTS
#
# RocketMQ already passed and will be reused/skipped.
#
# Reads only:
#   Results/Raw/<affected-project>/noise_XX__seed_YY/rankings.csv.gz
#   corresponding project_runs.csv for validation
#
# Writes only:
#   Results/Corrections/LatestFail_Sentinel_Fix_v2/
#
# No ML retraining.
# No frozen outputs modified.
# =============================================================================

ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
RESULTS = ROOT / "Results"
RAW_ROOT = RESULTS / "Raw"

CORRECTION_ROOT = (
    RESULTS
    / "Corrections"
    / "LatestFail_Sentinel_Fix_v2"
)
CORRECTION_ROOT.mkdir(parents=True, exist_ok=True)

AFFECTED_PROJECTS = [
    "EMResearch__EvoMaster",
    "JMRI__JMRI",
    "SonarSource__sonarqube",
    "apache__curator",
    "apache__logging-log4j2",
    "apache__rocketmq",
    "apache__shardingsphere",
    "apache__sling",
    "cantaloupe-project__cantaloupe",
    "eclipse__steady",
    "facebook__buck",
    "jcabi__jcabi-github",
    "yamcs__Yamcs",
    "zolyfarkas__spf4j",
]

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
SEEDS = list(range(1, 31))
EXPECTED_CONDITIONS = 270

TOL = 1e-10


# =============================================================================
# METRICS
# =============================================================================

def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)

    n = len(failures)
    m = int(failures.sum())

    if n == 0 or m == 0:
        return np.nan

    positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - positions.sum() / (n * m)
        + 1.0 / (2.0 * n)
    )


def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    n = len(failures)
    m = int(failures.sum())

    if n == 0 or m == 0:
        return np.nan

    total_duration = float(durations.sum())

    if (
        not np.isfinite(total_duration)
        or total_duration <= 0
    ):
        return np.nan

    cumulative_before = np.concatenate(
        [
            np.array([0.0]),
            np.cumsum(durations)[:-1],
        ]
    )

    mask = failures == 1

    midpoint_times = (
        cumulative_before[mask]
        + 0.5 * durations[mask]
    )

    return float(
        1.0
        - np.mean(
            midpoint_times / total_duration
        )
    )


def calculate_build_metrics(df, rank_col):
    result = []

    for build, g in df.groupby(
        "Build",
        sort=False
    ):
        g = g.sort_values(
            rank_col,
            kind="mergesort"
        )

        failures = int(
            g["CleanFailure"].sum()
        )

        # Same rule used by frozen experiment:
        # only failing evaluation builds are scored.
        if failures == 0:
            continue

        result.append(
            {
                "Build":
                    build,

                "Tests":
                    int(len(g)),

                "Failures":
                    failures,

                "TotalDuration":
                    float(
                        g["Duration"].sum()
                    ),

                "APFD":
                    calculate_apfd(
                        g["CleanFailure"]
                    ),

                "APFDc":
                    calculate_apfdc(
                        g["CleanFailure"],
                        g["Duration"],
                    ),
            }
        )

    return pd.DataFrame(result)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# =============================================================================
# CHECK FOR A COMPLETED CORRECTION
# =============================================================================

def completed_project_correction(project_slug):

    out = CORRECTION_ROOT / project_slug

    runs_path = (
        out
        / "LatestFail_corrected_project_runs.csv"
    )

    validation_path = (
        out
        / "LatestFail_correction_validation.csv"
    )

    manifest_path = (
        out
        / "LatestFail_correction_manifest.json"
    )

    # RocketMQ pilot used project-specific filenames.
    if project_slug == "apache__rocketmq":

        runs_path = (
            out
            / "ROCKETMQ_latestfail_corrected_project_runs.csv"
        )

        validation_path = (
            out
            / "ROCKETMQ_latestfail_correction_validation.csv"
        )

        manifest_path = (
            out
            / "ROCKETMQ_latestfail_correction_manifest.json"
        )

    if not (
        runs_path.is_file()
        and validation_path.is_file()
        and manifest_path.is_file()
    ):
        return False

    try:
        runs = pd.read_csv(runs_path)
        validation = pd.read_csv(validation_path)

        return (
            len(runs) == 270
            and len(validation) == 270
            and validation[
                "OldAPFDMatch"
            ].astype(bool).all()
            and validation[
                "OldAPFDcMatch"
            ].astype(bool).all()
        )

    except Exception:
        return False


# =============================================================================
# PROCESS ONE PROJECT
# =============================================================================

def correct_project(project_slug):

    raw_project = RAW_ROOT / project_slug

    if not raw_project.is_dir():
        raise FileNotFoundError(
            f"Missing raw project directory: {raw_project}"
        )

    out = CORRECTION_ROOT / project_slug
    out.mkdir(parents=True, exist_ok=True)

    validation_rows = []
    corrected_run_rows = []
    corrected_build_frames = []

    project_name = None
    project_number = None

    start = time.perf_counter()
    counter = 0

    for noise in NOISE_LEVELS:

        for seed in SEEDS:

            counter += 1

            condition_key = (
                f"noise_{noise:02d}__seed_{seed:02d}"
            )

            condition_dir = (
                raw_project
                / condition_key
            )

            ranking_path = (
                condition_dir
                / "rankings.csv.gz"
            )

            frozen_run_path = (
                condition_dir
                / "project_runs.csv"
            )

            if not ranking_path.is_file():
                raise FileNotFoundError(
                    f"Missing: {ranking_path}"
                )

            if not frozen_run_path.is_file():
                raise FileNotFoundError(
                    f"Missing: {frozen_run_path}"
                )

            # -------------------------------------------------------------
            # Read ONLY required ranking columns.
            # -------------------------------------------------------------

            rankings = pd.read_csv(
                ranking_path,
                usecols=[
                    "ProjectNumber",
                    "Project",
                    "ProjectSlug",
                    "ConditionKey",
                    "NoisePercent",
                    "RepetitionSeed",
                    "Technique",
                    "Build",
                    "Test",
                    "Rank",
                    "Score",
                    "CleanFailure",
                    "Duration",
                ],
                low_memory=False,
            )

            lf = rankings[
                rankings["Technique"].astype(str)
                == "LatestFail"
            ].copy()

            del rankings

            if lf.empty:
                raise RuntimeError(
                    f"{project_slug}/{condition_key}: "
                    "LatestFail rows missing"
                )

            # -------------------------------------------------------------
            # Metadata
            # -------------------------------------------------------------

            pnames = lf["Project"].drop_duplicates()
            pnums = lf["ProjectNumber"].drop_duplicates()

            if len(pnames) != 1 or len(pnums) != 1:
                raise RuntimeError(
                    f"{project_slug}/{condition_key}: "
                    "non-unique project metadata"
                )

            current_name = str(pnames.iloc[0])
            current_number = int(pnums.iloc[0])

            if project_name is None:
                project_name = current_name

            if project_number is None:
                project_number = current_number

            if current_name != project_name:
                raise RuntimeError(
                    "Project name changed between conditions"
                )

            if current_number != project_number:
                raise RuntimeError(
                    "Project number changed between conditions"
                )

            # -------------------------------------------------------------
            # Numeric normalization
            # -------------------------------------------------------------

            lf["Score"] = pd.to_numeric(
                lf["Score"],
                errors="raise"
            )

            lf["Rank"] = pd.to_numeric(
                lf["Rank"],
                errors="raise"
            ).astype(int)

            lf["CleanFailure"] = pd.to_numeric(
                lf["CleanFailure"],
                errors="raise"
            ).astype(int)

            lf["Duration"] = pd.to_numeric(
                lf["Duration"],
                errors="raise"
            ).astype(float)

            # -------------------------------------------------------------
            # Verify affected modern score convention.
            # -------------------------------------------------------------

            score_min = float(lf["Score"].min())
            score_max = float(lf["Score"].max())

            if score_max > 1.0 + 1e-12:
                raise RuntimeError(
                    f"{project_slug}/{condition_key}: "
                    f"unexpected score convention "
                    f"(min={score_min}, max={score_max})"
                )

            sentinel = np.isclose(
                lf["Score"].to_numpy(dtype=float),
                1.0,
                rtol=0,
                atol=1e-12,
            )

            lf["NeverFailed"] = sentinel

            sentinel_rows = int(
                sentinel.sum()
            )

            # -------------------------------------------------------------
            # OLD metrics reconstructed from frozen ranking.
            # -------------------------------------------------------------

            old_build = calculate_build_metrics(
                lf,
                "Rank"
            )

            if old_build.empty:
                raise RuntimeError(
                    f"{project_slug}/{condition_key}: "
                    "no scorable failing builds"
                )

            old_mean_apfd = float(
                old_build["APFD"].mean()
            )

            old_mean_apfdc = float(
                old_build["APFDc"].mean()
            )

            # -------------------------------------------------------------
            # Frozen project-run validation.
            # -------------------------------------------------------------

            frozen_runs = pd.read_csv(
                frozen_run_path,
                low_memory=False
            )

            frozen_lf = frozen_runs[
                frozen_runs["Technique"].astype(str)
                == "LatestFail"
            ]

            if len(frozen_lf) != 1:
                raise RuntimeError(
                    f"{project_slug}/{condition_key}: "
                    f"expected 1 frozen LatestFail row, "
                    f"found {len(frozen_lf)}"
                )

            frozen_lf = frozen_lf.iloc[0]

            frozen_apfd = float(
                frozen_lf["MeanAPFD"]
            )

            frozen_apfdc = float(
                frozen_lf["MeanAPFDc"]
            )

            old_apfd_match = np.isclose(
                old_mean_apfd,
                frozen_apfd,
                rtol=TOL,
                atol=1e-12,
            )

            old_apfdc_match = np.isclose(
                old_mean_apfdc,
                frozen_apfdc,
                rtol=TOL,
                atol=1e-12,
            )

            if not old_apfd_match:
                raise RuntimeError(
                    f"{project_slug}/{condition_key}: "
                    "old APFD reconstruction mismatch\n"
                    f"reconstructed={old_mean_apfd}\n"
                    f"frozen={frozen_apfd}"
                )

            if not old_apfdc_match:
                raise RuntimeError(
                    f"{project_slug}/{condition_key}: "
                    "old APFDc reconstruction mismatch\n"
                    f"reconstructed={old_mean_apfdc}\n"
                    f"frozen={frozen_apfdc}"
                )

            # -------------------------------------------------------------
            # CORRECTION
            # -------------------------------------------------------------

            corrected = lf.sort_values(
                [
                    "Build",
                    "NeverFailed",
                    "Score",
                    "Test",
                ],
                ascending=[
                    True,
                    True,     # real history before sentinel
                    False,    # most recent real failure first
                    True,     # deterministic tie-break
                ],
                kind="mergesort",
            ).copy()

            corrected["CorrectedRank"] = (
                corrected
                .groupby(
                    "Build",
                    sort=False
                )
                .cumcount()
                .add(1)
            )

            # -------------------------------------------------------------
            # Validate sentinel placement within every build.
            # -------------------------------------------------------------

            for build, g in corrected.groupby(
                "Build",
                sort=False
            ):
                flags = (
                    g["NeverFailed"]
                    .to_numpy(dtype=bool)
                )

                if np.any(
                    flags[:-1]
                    & (~flags[1:])
                ):
                    raise RuntimeError(
                        f"{project_slug}/"
                        f"{condition_key}/"
                        f"{build}: sentinel order failed"
                    )

            # -------------------------------------------------------------
            # Corrected build metrics.
            # -------------------------------------------------------------

            corrected_build = (
                calculate_build_metrics(
                    corrected,
                    "CorrectedRank"
                )
            )

            corrected_mean_apfd = float(
                corrected_build["APFD"].mean()
            )

            corrected_mean_apfdc = float(
                corrected_build["APFDc"].mean()
            )

            corrected_median_apfd = float(
                corrected_build["APFD"].median()
            )

            corrected_median_apfdc = float(
                corrected_build["APFDc"].median()
            )

            # -------------------------------------------------------------
            # Compact corrected build output.
            # -------------------------------------------------------------

            corrected_build.insert(
                0,
                "RepetitionSeed",
                seed
            )

            corrected_build.insert(
                0,
                "NoisePercent",
                noise
            )

            corrected_build.insert(
                0,
                "ConditionKey",
                condition_key
            )

            corrected_build.insert(
                0,
                "ProjectSlug",
                project_slug
            )

            corrected_build.insert(
                0,
                "Project",
                project_name
            )

            corrected_build.insert(
                0,
                "ProjectNumber",
                project_number
            )

            corrected_build.insert(
                6,
                "Technique",
                "LatestFail"
            )

            corrected_build_frames.append(
                corrected_build
            )

            # -------------------------------------------------------------
            # Project-run result.
            # -------------------------------------------------------------

            corrected_run_rows.append(
                {
                    "ProjectNumber":
                        project_number,

                    "Project":
                        project_name,

                    "ProjectSlug":
                        project_slug,

                    "ConditionKey":
                        condition_key,

                    "NoisePercent":
                        noise,

                    "RepetitionSeed":
                        seed,

                    "Technique":
                        "LatestFail",

                    "EvaluationBuilds":
                        int(
                            lf["Build"].nunique()
                        ),

                    "ScoredFailingBuilds":
                        int(
                            len(corrected_build)
                        ),

                    "EvaluationRows":
                        int(len(lf)),

                    "EvaluationFailures":
                        int(
                            lf[
                                "CleanFailure"
                            ].sum()
                        ),

                    "MeanAPFDc":
                        corrected_mean_apfdc,

                    "MedianAPFDc":
                        corrected_median_apfdc,

                    "MeanAPFD":
                        corrected_mean_apfd,

                    "MedianAPFD":
                        corrected_median_apfd,
                }
            )

            # -------------------------------------------------------------
            # Validation row.
            # -------------------------------------------------------------

            validation_rows.append(
                {
                    "ProjectNumber":
                        project_number,

                    "Project":
                        project_name,

                    "ProjectSlug":
                        project_slug,

                    "ConditionKey":
                        condition_key,

                    "NoisePercent":
                        noise,

                    "RepetitionSeed":
                        seed,

                    "LatestFailRows":
                        int(len(lf)),

                    "SentinelRows":
                        sentinel_rows,

                    "SentinelPercent":
                        (
                            100.0
                            * sentinel_rows
                            / len(lf)
                        ),

                    "FrozenMeanAPFD":
                        frozen_apfd,

                    "ReconstructedOldMeanAPFD":
                        old_mean_apfd,

                    "OldAPFDMatch":
                        bool(
                            old_apfd_match
                        ),

                    "CorrectedMeanAPFD":
                        corrected_mean_apfd,

                    "APFDChange":
                        (
                            corrected_mean_apfd
                            - frozen_apfd
                        ),

                    "FrozenMeanAPFDc":
                        frozen_apfdc,

                    "ReconstructedOldMeanAPFDc":
                        old_mean_apfdc,

                    "OldAPFDcMatch":
                        bool(
                            old_apfdc_match
                        ),

                    "CorrectedMeanAPFDc":
                        corrected_mean_apfdc,

                    "APFDcChange":
                        (
                            corrected_mean_apfdc
                            - frozen_apfdc
                        ),
                }
            )

            # Progress every 30 conditions.
            if counter % 30 == 0:

                elapsed = (
                    time.perf_counter()
                    - start
                )

                print(
                    f"    {counter:3d}/270 "
                    f"| noise={noise:02d}% "
                    f"| {elapsed:.1f}s",
                    flush=True,
                )

    # =========================================================================
    # PROJECT FINALIZATION
    # =========================================================================

    validation = pd.DataFrame(
        validation_rows
    ).sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
        ]
    ).reset_index(drop=True)

    corrected_runs = pd.DataFrame(
        corrected_run_rows
    ).sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
        ]
    ).reset_index(drop=True)

    corrected_builds = pd.concat(
        corrected_build_frames,
        ignore_index=True
    )

    if len(validation) != 270:
        raise RuntimeError(
            f"{project_slug}: "
            f"expected 270 validation rows, "
            f"got {len(validation)}"
        )

    if len(corrected_runs) != 270:
        raise RuntimeError(
            f"{project_slug}: "
            f"expected 270 corrected run rows, "
            f"got {len(corrected_runs)}"
        )

    if not validation[
        "OldAPFDMatch"
    ].all():
        raise RuntimeError(
            f"{project_slug}: APFD validation failed"
        )

    if not validation[
        "OldAPFDcMatch"
    ].all():
        raise RuntimeError(
            f"{project_slug}: APFDc validation failed"
        )

    # -------------------------------------------------------------------------
    # Per-project noise summary
    # -------------------------------------------------------------------------

    summary = (
        corrected_runs
        .groupby(
            "NoisePercent",
            as_index=False
        )
        .agg(
            Seeds=(
                "RepetitionSeed",
                "nunique",
            ),
            CorrectedMeanAPFDc=(
                "MeanAPFDc",
                "mean",
            ),
            CorrectedSDAPFDc=(
                "MeanAPFDc",
                "std",
            ),
            CorrectedMeanAPFD=(
                "MeanAPFD",
                "mean",
            ),
            CorrectedSDAPFD=(
                "MeanAPFD",
                "std",
            ),
        )
    )

    old_summary = (
        validation
        .groupby(
            "NoisePercent",
            as_index=False
        )
        .agg(
            FrozenMeanAPFDc=(
                "FrozenMeanAPFDc",
                "mean",
            ),
            FrozenMeanAPFD=(
                "FrozenMeanAPFD",
                "mean",
            ),
        )
    )

    summary = old_summary.merge(
        summary,
        on="NoisePercent",
        how="inner",
        validate="one_to_one",
    )

    summary["APFDcCorrection"] = (
        summary[
            "CorrectedMeanAPFDc"
        ]
        - summary[
            "FrozenMeanAPFDc"
        ]
    )

    summary["APFDCorrection"] = (
        summary[
            "CorrectedMeanAPFD"
        ]
        - summary[
            "FrozenMeanAPFD"
        ]
    )

    # -------------------------------------------------------------------------
    # Save
    # -------------------------------------------------------------------------

    validation_path = (
        out
        / "LatestFail_correction_validation.csv"
    )

    runs_path = (
        out
        / "LatestFail_corrected_project_runs.csv"
    )

    builds_path = (
        out
        / "LatestFail_corrected_build_metrics.csv"
    )

    summary_path = (
        out
        / "LatestFail_corrected_noise_summary.csv"
    )

    manifest_path = (
        out
        / "LatestFail_correction_manifest.json"
    )

    validation.to_csv(
        validation_path,
        index=False
    )

    corrected_runs.to_csv(
        runs_path,
        index=False
    )

    corrected_builds.to_csv(
        builds_path,
        index=False
    )

    summary.to_csv(
        summary_path,
        index=False
    )

    manifest = {
        "CorrectionID":
            "LatestFail_Sentinel_Fix_v2",

        "ProjectNumber":
            int(project_number),

        "Project":
            project_name,

        "ProjectSlug":
            project_slug,

        "CreatedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "ConditionsProcessed":
            270,

        "FrozenSourceModification":
            False,

        "OldAPFDValidationPass":
            True,

        "OldAPFDcValidationPass":
            True,

        "Correction":
            (
                "REC_LastFailureAge=-1 denotes no prior "
                "failure. The affected implementation "
                "negated this sentinel to Score=+1 and "
                "ranked it ahead of genuine prior failures. "
                "Sentinel rows are corrected to lowest "
                "LatestFail priority."
            ),

        "ValidationSHA256":
            sha256_file(
                validation_path
            ),

        "CorrectedRunsSHA256":
            sha256_file(
                runs_path
            ),

        "CorrectedBuildsSHA256":
            sha256_file(
                builds_path
            ),

        "NoiseSummarySHA256":
            sha256_file(
                summary_path
            ),
    }

    with open(
        manifest_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            manifest,
            f,
            indent=2,
            sort_keys=True,
        )

    elapsed = (
        time.perf_counter()
        - start
    )

    return {
        "Project":
            project_name,

        "ProjectSlug":
            project_slug,

        "Seconds":
            elapsed,

        "Conditions":
            len(corrected_runs),

        "ValidationAPFD":
            True,

        "ValidationAPFDc":
            True,

        "CleanFrozenAPFDc":
            float(
                summary.loc[
                    summary[
                        "NoisePercent"
                    ] == 0,
                    "FrozenMeanAPFDc"
                ].iloc[0]
            ),

        "CleanCorrectedAPFDc":
            float(
                summary.loc[
                    summary[
                        "NoisePercent"
                    ] == 0,
                    "CorrectedMeanAPFDc"
                ].iloc[0]
            ),
    }


# =============================================================================
# EXECUTE / RESUME
# =============================================================================

print("=" * 90)
print("LATESTFAIL SENTINEL FIX V2 — AFFECTED PROJECTS")
print("=" * 90)

job_results = []

for i, project_slug in enumerate(
    AFFECTED_PROJECTS,
    start=1
):

    print(
        f"\n[{i:02d}/14] {project_slug}",
        flush=True
    )

    if completed_project_correction(
        project_slug
    ):
        print(
            "    PASS already corrected — SKIP",
            flush=True
        )

        job_results.append(
            {
                "ProjectSlug":
                    project_slug,

                "Status":
                    "ALREADY_COMPLETE",
            }
        )

        continue

    result = correct_project(
        project_slug
    )

    print(
        f"    PASS | "
        f"{result['Seconds']:.1f}s | "
        f"clean APFDc "
        f"{result['CleanFrozenAPFDc']:.4f}"
        f" -> "
        f"{result['CleanCorrectedAPFDc']:.4f}",
        flush=True
    )

    result["Status"] = "CORRECTED"
    job_results.append(result)


print("\n" + "=" * 90)
print("CORRECTION JOB FINISHED")
print("=" * 90)

job_df = pd.DataFrame(job_results)
display(job_df)

print("\nCorrection root:")
print(CORRECTION_ROOT)

LATESTFAIL SENTINEL FIX V2 — AFFECTED PROJECTS

[01/14] EMResearch__EvoMaster
    PASS already corrected — SKIP

[02/14] JMRI__JMRI
     30/270 | noise=00% | 76.2s
     60/270 | noise=05% | 138.0s
     90/270 | noise=10% | 203.1s
    120/270 | noise=15% | 278.1s
    150/270 | noise=20% | 352.5s
    180/270 | noise=25% | 427.0s
    210/270 | noise=30% | 512.0s
    240/270 | noise=40% | 590.3s
    270/270 | noise=50% | 667.3s
    PASS | 667.5s | clean APFDc 0.3170 -> 0.7459

[03/14] SonarSource__sonarqube
     30/270 | noise=00% | 23.5s
     60/270 | noise=05% | 46.2s
     90/270 | noise=10% | 68.9s
    120/270 | noise=15% | 92.1s
    150/270 | noise=20% | 114.1s
    180/270 | noise=25% | 136.2s
    210/270 | noise=30% | 158.2s
    240/270 | noise=40% | 180.5s
    270/270 | noise=50% | 203.7s
    PASS | 203.9s | clean APFDc 0.5289 -> 0.5225

[04/14] apache__curator
     30/270 | noise=00% | 12.3s
     60/270 | noise=05% | 23.9s
     90/270 | noise=10% | 35.6s
    120/270 | noise=15% | 47

,ProjectSlug,Status,Project,Seconds,Conditions,ValidationAPFD,ValidationAPFDc,CleanFrozenAPFDc,CleanCorrectedAPFDc
0,EMResearch__EvoMaster,ALREADY_COMPLETE,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JMRI__JMRI,CORRECTED,JMRI@JMRI,667.529225,270.0,True,True,0.316958,0.745879
2,SonarSource__sonarqube,CORRECTED,SonarSource@sonarqube,203.948539,270.0,True,True,0.528929,0.522529
3,apache__curator,CORRECTED,apache@curator,110.295692,270.0,True,True,0.352857,0.672631
4,apache__logging-log4j2,CORRECTED,apache@logging-log4j2,260.110689,270.0,True,True,0.048835,0.914058
5,apache__rocketmq,ALREADY_COMPLETE,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,apache__shardingsphere,CORRECTED,apache@shardingsphere,173.058173,270.0,True,True,0.117362,0.972209
7,apache__sling,CORRECTED,apache@sling,134.111911,270.0,True,True,0.364407,0.909743
8,cantaloupe-project__cantaloupe,CORRECTED,cantaloupe-project@cantaloupe,172.095191,270.0,True,True,0.569503,0.497035
9,eclipse__steady,CORRECTED,eclipse@steady,169.159141,270.0,True,True,0.294815,0.964422



Correction root:
/content/drive/MyDrive/Thesis_Experiment/Results/Corrections/LatestFail_Sentinel_Fix_v2


In [16]:
from pathlib import Path
from datetime import datetime, timezone
import os
import json
import hashlib

import pandas as pd
import numpy as np

# ============================================================================
# STEP 4 — CONSOLIDATE FINAL LATESTFAIL ACROSS ALL 24 PROJECTS
#
# 14 affected projects:
#     use corrected V2 project-run results
#
# 10 legacy projects:
#     preserve original LatestFail results unchanged
#
# Also reconstruct the ORIGINAL 24-project curve for comparison.
#
# No rankings are opened.
# No ML results are changed.
# No frozen source is modified.
# ============================================================================

ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
RESULTS = ROOT / "Results"
RAW_ROOT = RESULTS / "Raw"

CORRECTION_ROOT = (
    RESULTS
    / "Corrections"
    / "LatestFail_Sentinel_Fix_v2"
)

GLOBAL_OUT = (
    CORRECTION_ROOT
    / "global_24_project_consolidation"
)

GLOBAL_OUT.mkdir(
    parents=True,
    exist_ok=True
)


AFFECTED = [
    "EMResearch__EvoMaster",
    "JMRI__JMRI",
    "SonarSource__sonarqube",
    "apache__curator",
    "apache__logging-log4j2",
    "apache__rocketmq",
    "apache__shardingsphere",
    "apache__sling",
    "cantaloupe-project__cantaloupe",
    "eclipse__steady",
    "facebook__buck",
    "jcabi__jcabi-github",
    "yamcs__Yamcs",
    "zolyfarkas__spf4j",
]


LEGACY_UNCHANGED = [
    "Angel-ML__angel",
    "CompEvol__beast2",
    "apache__airavata",
    "b2ihealthcare__snow-owl",
    "camunda__camunda-bpm-platform",
    "eclipse__jetty.project",
    "eclipse__paho.mqtt.java",
    "optimatika__ojAlgo",
    "spring-cloud__spring-cloud-dataflow",
    "thinkaurelius__titan",
]


NOISE_LEVELS = [
    0, 5, 10, 15, 20,
    25, 30, 40, 50
]

SEEDS = list(range(1, 31))

EXPECTED_ROWS_PER_PROJECT = 270
EXPECTED_GLOBAL_ROWS = 24 * 270


# ============================================================================
# HELPERS
# ============================================================================

def sha256_file(path, chunk_size=8 * 1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def validate_grid(df, project_slug, label):

    if len(df) != 270:
        raise RuntimeError(
            f"{project_slug} [{label}]: "
            f"expected 270 rows, found {len(df)}"
        )

    pairs = set(
        zip(
            pd.to_numeric(
                df["NoisePercent"]
            ).astype(int),

            pd.to_numeric(
                df["RepetitionSeed"]
            ).astype(int),
        )
    )

    expected = {
        (noise, seed)
        for noise in NOISE_LEVELS
        for seed in SEEDS
    }

    if pairs != expected:

        missing = sorted(
            expected - pairs
        )

        extra = sorted(
            pairs - expected
        )

        raise RuntimeError(
            f"{project_slug} [{label}] "
            f"invalid noise/seed grid.\n"
            f"Missing: {missing[:20]}\n"
            f"Extra: {extra[:20]}"
        )


def normalized_rows(
    df,
    project_slug,
    source_type,
):

    required = {
        "NoisePercent",
        "RepetitionSeed",
        "MeanAPFD",
        "MeanAPFDc",
    }

    missing = (
        required - set(df.columns)
    )

    if missing:
        raise RuntimeError(
            f"{project_slug}: missing columns {missing}"
        )

    out = pd.DataFrame({
        "ProjectSlug":
            project_slug,

        "NoisePercent":
            pd.to_numeric(
                df["NoisePercent"],
                errors="raise"
            ).astype(int),

        "RepetitionSeed":
            pd.to_numeric(
                df["RepetitionSeed"],
                errors="raise"
            ).astype(int),

        "MeanAPFD":
            pd.to_numeric(
                df["MeanAPFD"],
                errors="raise"
            ).astype(float),

        "MeanAPFDc":
            pd.to_numeric(
                df["MeanAPFDc"],
                errors="raise"
            ).astype(float),

        "SourceType":
            source_type,
    })

    return out


# ============================================================================
# 1. LOAD AFFECTED PROJECTS
#
# For each affected project:
#   FINAL    = corrected project-run result
#   ORIGINAL = frozen result copied into correction validation
# ============================================================================

final_frames = []
original_frames = []

print("=" * 90)
print("LOADING 14 CORRECTED PROJECTS")
print("=" * 90)


for i, slug in enumerate(
    AFFECTED,
    start=1
):

    out = CORRECTION_ROOT / slug

    if slug == "apache__rocketmq":

        corrected_path = (
            out
            / "ROCKETMQ_latestfail_corrected_project_runs.csv"
        )

        validation_path = (
            out
            / "ROCKETMQ_latestfail_correction_validation.csv"
        )

    else:

        corrected_path = (
            out
            / "LatestFail_corrected_project_runs.csv"
        )

        validation_path = (
            out
            / "LatestFail_correction_validation.csv"
        )

    if not corrected_path.is_file():
        raise FileNotFoundError(
            corrected_path
        )

    if not validation_path.is_file():
        raise FileNotFoundError(
            validation_path
        )

    corrected = pd.read_csv(
        corrected_path
    )

    validation = pd.read_csv(
        validation_path
    )

    validate_grid(
        corrected,
        slug,
        "CORRECTED"
    )

    validate_grid(
        validation,
        slug,
        "VALIDATION"
    )

    if not validation[
        "OldAPFDMatch"
    ].astype(bool).all():

        raise RuntimeError(
            f"{slug}: old APFD validation not all TRUE"
        )

    if not validation[
        "OldAPFDcMatch"
    ].astype(bool).all():

        raise RuntimeError(
            f"{slug}: old APFDc validation not all TRUE"
        )

    final_frames.append(
        normalized_rows(
            corrected,
            slug,
            "CORRECTED_MODERN",
        )
    )

    # Build original/frozen project-run rows from validation file.
    original = pd.DataFrame({
        "NoisePercent":
            validation["NoisePercent"],

        "RepetitionSeed":
            validation["RepetitionSeed"],

        "MeanAPFD":
            validation["FrozenMeanAPFD"],

        "MeanAPFDc":
            validation["FrozenMeanAPFDc"],
    })

    original_frames.append(
        normalized_rows(
            original,
            slug,
            "ORIGINAL_AFFECTED",
        )
    )

    print(
        f"[{i:02d}/14] {slug}: PASS"
    )


# ============================================================================
# 2. LOAD 10 LEGACY PROJECTS UNCHANGED
#
# Reads ONLY project_run_metrics.csv or project_run.csv.
# No ranking files are opened.
# ============================================================================

print("\n" + "=" * 90)
print("LOADING 10 LEGACY / UNCHANGED PROJECTS")
print("=" * 90)


for i, slug in enumerate(
    LEGACY_UNCHANGED,
    start=1
):

    project_root = (
        RAW_ROOT / slug
    )

    if not project_root.is_dir():
        raise FileNotFoundError(
            project_root
        )

    project_rows = []

    # Locate the 270 small project-run files.
    run_paths = []

    for current_root, dirs, files in os.walk(
        project_root
    ):

        root = Path(current_root)

        if "project_run_metrics.csv" in files:

            run_paths.append(
                root
                / "project_run_metrics.csv"
            )

        elif "project_run.csv" in files:

            run_paths.append(
                root
                / "project_run.csv"
            )

    if len(run_paths) != 270:

        raise RuntimeError(
            f"{slug}: expected 270 legacy "
            f"project-run files, found {len(run_paths)}"
        )

    for n, path in enumerate(
        sorted(run_paths),
        start=1
    ):

        # These files are very small.
        df = pd.read_csv(
            path,
            usecols=[
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
                "MeanAPFD",
                "MeanAPFDc",
            ],
        )

        lf = df[
            df["Technique"]
            .astype(str)
            .str.lower()
            == "latestfail"
        ]

        if len(lf) != 1:

            raise RuntimeError(
                f"{slug}: {path} "
                f"expected 1 LatestFail row, "
                f"found {len(lf)}"
            )

        project_rows.append(
            lf[
                [
                    "NoisePercent",
                    "RepetitionSeed",
                    "MeanAPFD",
                    "MeanAPFDc",
                ]
            ].iloc[0].to_dict()
        )

        if n % 90 == 0:

            print(
                f"    {slug}: "
                f"{n}/270 files",
                flush=True
            )

    legacy = pd.DataFrame(
        project_rows
    )

    validate_grid(
        legacy,
        slug,
        "LEGACY"
    )

    normalized = normalized_rows(
        legacy,
        slug,
        "UNCHANGED_LEGACY",
    )

    # Legacy result is identical in original and corrected analyses.
    final_frames.append(
        normalized.copy()
    )

    original_frames.append(
        normalized.copy()
    )

    print(
        f"[{i:02d}/10] {slug}: PASS",
        flush=True
    )


# ============================================================================
# 3. ASSEMBLE 24-PROJECT ORIGINAL AND FINAL TABLES
# ============================================================================

final_all = pd.concat(
    final_frames,
    ignore_index=True
)

original_all = pd.concat(
    original_frames,
    ignore_index=True
)


if len(final_all) != EXPECTED_GLOBAL_ROWS:

    raise RuntimeError(
        f"FINAL: expected {EXPECTED_GLOBAL_ROWS} rows, "
        f"found {len(final_all)}"
    )


if len(original_all) != EXPECTED_GLOBAL_ROWS:

    raise RuntimeError(
        f"ORIGINAL: expected {EXPECTED_GLOBAL_ROWS} rows, "
        f"found {len(original_all)}"
    )


if final_all[
    "ProjectSlug"
].nunique() != 24:

    raise RuntimeError(
        "FINAL result does not contain 24 projects"
    )


if original_all[
    "ProjectSlug"
].nunique() != 24:

    raise RuntimeError(
        "ORIGINAL result does not contain 24 projects"
    )


# Validate every project independently.
for slug, g in final_all.groupby(
    "ProjectSlug"
):

    validate_grid(
        g,
        slug,
        "FINAL_GLOBAL"
    )


for slug, g in original_all.groupby(
    "ProjectSlug"
):

    validate_grid(
        g,
        slug,
        "ORIGINAL_GLOBAL"
    )


# ============================================================================
# 4. PROJECT × NOISE AGGREGATION
#
# First average 30 seeds within each project.
# Then average projects equally.
# ============================================================================

def project_noise_table(df):

    return (
        df
        .groupby(
            [
                "ProjectSlug",
                "NoisePercent",
            ],
            as_index=False
        )
        .agg(
            Seeds=(
                "RepetitionSeed",
                "nunique",
            ),

            ProjectMeanAPFDc=(
                "MeanAPFDc",
                "mean",
            ),

            ProjectMeanAPFD=(
                "MeanAPFD",
                "mean",
            ),
        )
        .sort_values(
            [
                "ProjectSlug",
                "NoisePercent",
            ]
        )
        .reset_index(drop=True)
    )


original_project_noise = (
    project_noise_table(
        original_all
    )
)

final_project_noise = (
    project_noise_table(
        final_all
    )
)


assert len(original_project_noise) == 24 * 9
assert len(final_project_noise) == 24 * 9

assert (
    original_project_noise["Seeds"]
    == 30
).all()

assert (
    final_project_noise["Seeds"]
    == 30
).all()


# ============================================================================
# 5. GLOBAL NOISE SUMMARY
# ============================================================================

def global_summary(
    project_noise,
    prefix,
):

    return (
        project_noise
        .groupby(
            "NoisePercent",
            as_index=False
        )
        .agg(
            Projects=(
                "ProjectSlug",
                "nunique",
            ),

            **{
                f"{prefix}MeanAPFDc":
                    (
                        "ProjectMeanAPFDc",
                        "mean",
                    ),

                f"{prefix}SDProjectsAPFDc":
                    (
                        "ProjectMeanAPFDc",
                        "std",
                    ),

                f"{prefix}MeanAPFD":
                    (
                        "ProjectMeanAPFD",
                        "mean",
                    ),

                f"{prefix}SDProjectsAPFD":
                    (
                        "ProjectMeanAPFD",
                        "std",
                    ),
            }
        )
        .sort_values(
            "NoisePercent"
        )
        .reset_index(drop=True)
    )


old_global = global_summary(
    original_project_noise,
    "Original"
)

new_global = global_summary(
    final_project_noise,
    "Corrected"
)


global_compare = old_global.merge(
    new_global.drop(
        columns=["Projects"]
    ),
    on="NoisePercent",
    how="inner",
    validate="one_to_one",
)


global_compare[
    "APFDcDifference"
] = (
    global_compare[
        "CorrectedMeanAPFDc"
    ]
    - global_compare[
        "OriginalMeanAPFDc"
    ]
)


global_compare[
    "APFDDifference"
] = (
    global_compare[
        "CorrectedMeanAPFD"
    ]
    - global_compare[
        "OriginalMeanAPFD"
    ]
)


# ============================================================================
# 6. CORRECTION-SCOPE SUMMARY
# ============================================================================

scope_rows = []

for slug in sorted(
    set(AFFECTED + LEGACY_UNCHANGED)
):

    old_p = original_project_noise[
        original_project_noise[
            "ProjectSlug"
        ] == slug
    ]

    new_p = final_project_noise[
        final_project_noise[
            "ProjectSlug"
        ] == slug
    ]

    merged = old_p.merge(
        new_p,
        on=[
            "ProjectSlug",
            "NoisePercent"
        ],
        suffixes=(
            "_Original",
            "_Corrected"
        ),
        how="inner",
        validate="one_to_one",
    )

    clean = merged[
        merged["NoisePercent"] == 0
    ].iloc[0]

    scope_rows.append({
        "ProjectSlug":
            slug,

        "Status":
            (
                "CORRECTED"
                if slug in AFFECTED
                else "UNCHANGED_LEGACY"
            ),

        "OriginalCleanAPFDc":
            clean[
                "ProjectMeanAPFDc_Original"
            ],

        "CorrectedCleanAPFDc":
            clean[
                "ProjectMeanAPFDc_Corrected"
            ],

        "CleanAPFDcDifference":
            (
                clean[
                    "ProjectMeanAPFDc_Corrected"
                ]
                -
                clean[
                    "ProjectMeanAPFDc_Original"
                ]
            ),
    })


scope = pd.DataFrame(
    scope_rows
).sort_values(
    [
        "Status",
        "ProjectSlug",
    ]
).reset_index(drop=True)


# ============================================================================
# 7. SAVE AUTHORITATIVE CORRECTED LATESTFAIL PACKAGE
# ============================================================================

FINAL_RUNS_PATH = (
    GLOBAL_OUT
    / "LatestFail_FINAL_project_runs_24_projects.csv"
)

ORIGINAL_RUNS_PATH = (
    GLOBAL_OUT
    / "LatestFail_ORIGINAL_project_runs_24_projects.csv"
)

FINAL_PROJECT_NOISE_PATH = (
    GLOBAL_OUT
    / "LatestFail_FINAL_project_noise_24_projects.csv"
)

ORIGINAL_PROJECT_NOISE_PATH = (
    GLOBAL_OUT
    / "LatestFail_ORIGINAL_project_noise_24_projects.csv"
)

GLOBAL_COMPARE_PATH = (
    GLOBAL_OUT
    / "LatestFail_original_vs_corrected_global_summary.csv"
)

SCOPE_PATH = (
    GLOBAL_OUT
    / "LatestFail_correction_scope.csv"
)

MANIFEST_PATH = (
    GLOBAL_OUT
    / "LATESTFAIL_GLOBAL_CORRECTION_MANIFEST.json"
)


final_all.to_csv(
    FINAL_RUNS_PATH,
    index=False
)

original_all.to_csv(
    ORIGINAL_RUNS_PATH,
    index=False
)

final_project_noise.to_csv(
    FINAL_PROJECT_NOISE_PATH,
    index=False
)

original_project_noise.to_csv(
    ORIGINAL_PROJECT_NOISE_PATH,
    index=False
)

global_compare.to_csv(
    GLOBAL_COMPARE_PATH,
    index=False
)

scope.to_csv(
    SCOPE_PATH,
    index=False
)


manifest = {
    "CorrectionID":
        "LatestFail_Sentinel_Fix_v2",

    "CreatedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "Projects":
        24,

    "AffectedProjectsCorrected":
        14,

    "LegacyProjectsPreserved":
        10,

    "ConditionsPerProject":
        270,

    "FinalProjectRunRows":
        int(len(final_all)),

    "FinalProjectNoiseRows":
        int(len(final_project_noise)),

    "FrozenSourceModification":
        False,

    "Aggregation":
        (
            "Thirty repetition seeds are first averaged "
            "within each project and noise level; "
            "the resulting 24 project means are then "
            "averaged with equal project weighting."
        ),

    "FinalRunsSHA256":
        sha256_file(
            FINAL_RUNS_PATH
        ),

    "OriginalRunsSHA256":
        sha256_file(
            ORIGINAL_RUNS_PATH
        ),

    "FinalProjectNoiseSHA256":
        sha256_file(
            FINAL_PROJECT_NOISE_PATH
        ),

    "GlobalComparisonSHA256":
        sha256_file(
            GLOBAL_COMPARE_PATH
        ),

    "CorrectionScopeSHA256":
        sha256_file(
            SCOPE_PATH
        ),
}


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        sort_keys=True,
    )


# ============================================================================
# FINAL REPORT
# ============================================================================

print("\n" + "=" * 90)
print("LATESTFAIL 24-PROJECT CONSOLIDATION — PASS")
print("=" * 90)

print(
    "Final project-run rows:",
    len(final_all)
)

print(
    "Original project-run rows:",
    len(original_all)
)

print(
    "Projects:",
    final_all["ProjectSlug"].nunique()
)

print(
    "Final project-noise rows:",
    len(final_project_noise)
)

print(
    "Corrected projects:",
    len(AFFECTED)
)

print(
    "Unchanged legacy projects:",
    len(LEGACY_UNCHANGED)
)


print(
    "\n=== ORIGINAL VS CORRECTED GLOBAL LATESTFAIL ==="
)

display(
    global_compare
)


print(
    "\n=== CLEAN-CONDITION CORRECTION BY PROJECT ==="
)

display(
    scope
)


print("\nSaved to:")
print(GLOBAL_OUT)

LOADING 14 CORRECTED PROJECTS
[01/14] EMResearch__EvoMaster: PASS
[02/14] JMRI__JMRI: PASS
[03/14] SonarSource__sonarqube: PASS
[04/14] apache__curator: PASS
[05/14] apache__logging-log4j2: PASS
[06/14] apache__rocketmq: PASS
[07/14] apache__shardingsphere: PASS
[08/14] apache__sling: PASS
[09/14] cantaloupe-project__cantaloupe: PASS
[10/14] eclipse__steady: PASS
[11/14] facebook__buck: PASS
[12/14] jcabi__jcabi-github: PASS
[13/14] yamcs__Yamcs: PASS
[14/14] zolyfarkas__spf4j: PASS

LOADING 10 LEGACY / UNCHANGED PROJECTS
    Angel-ML__angel: 90/270 files
    Angel-ML__angel: 180/270 files
    Angel-ML__angel: 270/270 files
[01/10] Angel-ML__angel: PASS
    CompEvol__beast2: 90/270 files
    CompEvol__beast2: 180/270 files
    CompEvol__beast2: 270/270 files
[02/10] CompEvol__beast2: PASS
    apache__airavata: 90/270 files
    apache__airavata: 180/270 files
    apache__airavata: 270/270 files
[03/10] apache__airavata: PASS
    b2ihealthcare__snow-owl: 90/270 files
    b2ihealthcare__s

,NoisePercent,Projects,OriginalMeanAPFDc,OriginalSDProjectsAPFDc,OriginalMeanAPFD,OriginalSDProjectsAPFD,CorrectedMeanAPFDc,CorrectedSDProjectsAPFDc,CorrectedMeanAPFD,CorrectedSDProjectsAPFD,APFDcDifference,APFDDifference
0,0,24,0.575256,0.255355,0.509353,0.352934,0.769589,0.155050,0.858287,0.119240,0.194332,0.348934
1,5,24,0.783330,0.122301,0.833622,0.107858,0.769982,0.139440,0.825690,0.122206,-0.013347,-0.007932
2,10,24,0.783453,0.124296,0.833985,0.107063,0.770568,0.139746,0.824664,0.122049,-0.012885,-0.009321
3,15,24,0.783334,0.124556,0.833737,0.107159,0.770503,0.139600,0.823996,0.122073,-0.012831,-0.009741
4,20,24,0.782989,0.124783,0.834001,0.105550,0.770250,0.139940,0.824248,0.121029,-0.012739,-0.009753
5,25,24,0.783038,0.127367,0.835076,0.105282,0.770251,0.142211,0.825210,0.120361,-0.012788,-0.009866
6,30,24,0.782263,0.128647,0.834789,0.105236,0.769684,0.142816,0.825116,0.119768,-0.012579,-0.009673
7,40,24,0.780254,0.129072,0.835860,0.100364,0.767779,0.142759,0.826272,0.115210,-0.012475,-0.009589
8,50,24,0.779787,0.130316,0.835995,0.101855,0.767383,0.144215,0.826499,0.116545,-0.012405,-0.009496



=== CLEAN-CONDITION CORRECTION BY PROJECT ===


,ProjectSlug,Status,OriginalCleanAPFDc,CorrectedCleanAPFDc,CleanAPFDcDifference
0,EMResearch__EvoMaster,CORRECTED,0.689091,0.716793,0.027702
1,JMRI__JMRI,CORRECTED,0.316958,0.745879,0.428921
2,SonarSource__sonarqube,CORRECTED,0.528929,0.522529,-0.006400
3,apache__curator,CORRECTED,0.352857,0.672631,0.319773
4,apache__logging-log4j2,CORRECTED,0.048835,0.914058,0.865223
5,apache__rocketmq,CORRECTED,0.299580,0.894954,0.595373
6,apache__shardingsphere,CORRECTED,0.117362,0.972209,0.854847
7,apache__sling,CORRECTED,0.364407,0.909743,0.545337
8,cantaloupe-project__cantaloupe,CORRECTED,0.569503,0.497035,-0.072468
9,eclipse__steady,CORRECTED,0.294815,0.964422,0.669608



Saved to:
/content/drive/MyDrive/Thesis_Experiment/Results/Corrections/LatestFail_Sentinel_Fix_v2/global_24_project_consolidation


In [1]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

out_dir = Path("/mnt/data/thesis_rq2_corrected_figures")
out_dir.mkdir(parents=True, exist_ok=True)

noise = np.array([0, 5, 10, 15, 20, 25, 30, 40, 50])

latest_apfdc = np.array([
    0.769589, 0.769982, 0.770568, 0.770503, 0.770250,
    0.770251, 0.769684, 0.767779, 0.767383
])
latest_apfd = np.array([
    0.858287, 0.825690, 0.824664, 0.823996, 0.824248,
    0.825210, 0.825116, 0.826272, 0.826499
])

apfdc = {
    "Random Forest": np.array([0.803239, 0.674281, 0.625262, 0.591190, 0.567499, 0.543888, 0.521425, 0.479389, 0.443023]),
    "XGBoost": np.array([0.784217, 0.684539, 0.641205, 0.609067, 0.579711, 0.548557, 0.528129, 0.487266, 0.426752]),
    "LightGBM": np.array([0.762060, 0.680160, 0.647043, 0.627722, 0.598752, 0.572690, 0.547930, 0.498369, 0.432476]),
    "Naive Bayes": np.array([0.617603, 0.573381, 0.558295, 0.544015, 0.546060, 0.541469, 0.537718, 0.523996, 0.492052]),
}

apfd = {
    "Random Forest": np.array([0.903964, 0.746524, 0.670719, 0.634206, 0.603619, 0.578378, 0.549337, 0.491977, 0.432852]),
    "XGBoost": np.array([0.917151, 0.762095, 0.713787, 0.673993, 0.635888, 0.608000, 0.583909, 0.529730, 0.455526]),
    "LightGBM": np.array([0.870486, 0.753960, 0.714033, 0.689077, 0.653984, 0.630353, 0.598944, 0.542784, 0.456621]),
    "Naive Bayes": np.array([0.786290, 0.804332, 0.771700, 0.732417, 0.715936, 0.693917, 0.668810, 0.606167, 0.489391]),
}

markers = ["o", "s", "^", "D"]

def make_plot(model_values, baseline, ylabel, filename):
    fig, ax = plt.subplots(figsize=(8.3, 4.9))
    for (name, vals), marker in zip(model_values.items(), markers):
        advantage = vals - baseline
        ax.plot(noise, advantage, marker=marker, linewidth=1.8, markersize=5, label=name)
    ax.axhline(0, linestyle="--", linewidth=1.2)
    ax.set_xlabel("Simulated training-verdict noise (%)")
    ax.set_ylabel(ylabel)
    ax.set_xticks(noise)
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    path = out_dir / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return path

p1 = make_plot(
    apfdc,
    latest_apfdc,
    "Mean APFDc advantage over LatestFail",
    "Figure_5_4_corrected_APFDc_advantage_vs_LatestFail.png"
)

p2 = make_plot(
    apfd,
    latest_apfd,
    "Mean APFD advantage over LatestFail",
    "Figure_5_5_corrected_APFD_advantage_vs_LatestFail.png"
)

print(f"Created:\n{p1}\n{p2}")


Created:
/mnt/data/thesis_rq2_corrected_figures/Figure_5_4_corrected_APFDc_advantage_vs_LatestFail.png
/mnt/data/thesis_rq2_corrected_figures/Figure_5_5_corrected_APFD_advantage_vs_LatestFail.png
